# Paso 1: Recuperar y verificar los componentes del avance de proyecto.

## Tabla de trazabilidad — Avance de Proyecto

| Elemento | Archivo o ubicación | Versión | Estado | Acción requerida |
|---|---|---|---|---|
| Dataset | Se carga vía `kagglehub` (`blastchar/telco-customer-churn`), no hay CSV local en el repo | Consultado 25–31 ago 2026 | Disponible (requiere `kagglehub` y conexión) | Documentar que requiere autenticación/conexión a Kaggle |
| Pipeline | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 13) | ColumnTransformer con imputación (mediana), StandardScaler, OneHotEncoder, OrdinalEncoder | Corre limpio | Ninguna |
| Baseline | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 14) | DummyClassifier (most_frequent) — Acc 0.7346, Recall 0.0000 | Completo | Ninguna |
| Modelo preliminar | `Avance_Proyecto/Avance_Proyecto.ipynb` (Pasos 15–19) | Regresión Logística (max_iter=1000, solver='lbfgs') — Acc 0.8055, F1 0.6040, ROC-AUC 0.8421 | Seleccionado y justificado | Ninguna |
| Métricas | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 16) | Accuracy, Precision, Recall, F1, ROC-AUC para ambos modelos | Completo | Ninguna |
| Informe del avance de proyecto | `Avance_Proyecto/Avance_Proyecto.ipynb` (87 celdas) | Ejecutado 31 ago 2026, Python 3.14.7 | Corre de inicio a fin sin errores | Ninguna |

# Paso 2: Documentar la evolución del proyecto.

| Elemento modificado | Situación en avance de proyecto | Cambio realizado | Justificación | Impacto esperado |
|---|---|---|---|---|
| Datos | Telco Customer Churn (Kaggle, 7043 registros, 21 variables), cargado en tiempo de ejecución vía `kagglehub`. TotalCharges tenía tipo incorrecto (str) con 11 valores nulos ocultos, corregido a float64. Sin duplicados. Variable objetivo `Churn` con 26.5% de abandono (clase desbalanceada). | Sin cambios en el dataset base. Se repitió la revisión de calidad (nulos, duplicados, tipos, categorías, rangos, variables excluidas, balance de clases, fuga de información y correspondencia columnas-pipeline). | Verificar que los datos conservan las condiciones del avance antes de continuar, para que los resultados sean comparables. | Se confirma que los datos son consistentes: 0 nulos, 0 duplicados, sin valores fuera de rango y balance 73.46% / 26.54%. |
| Preprocesamiento | ColumnTransformer con: imputación por mediana en numéricas, StandardScaler en tenure/MonthlyCharges/TotalCharges, OneHotEncoder en categóricas nominales, OrdinalEncoder en Contract. Partición 70/15/15 (train/val/test) estratificada, semilla 42. | `SeniorCitizen` y `PaperlessBilling` no estaban asignadas a ningún grupo del ColumnTransformer y pasaban por `remainder='passthrough'`. Se agregaron a `cols_bin` (imputación + OrdinalEncoder). El pipeline final incorpora además las variables nuevas `Total_Servicios` y `Cliente_Nuevo`. | La revisión de correspondencia columnas-pipeline detectó variables sin transformación explícita. | Todas las variables pasan por imputación y codificación controlada, sin variables sin procesar. |
| Modelos | Baseline: DummyClassifier (most_frequent). Regresión Logística (max_iter=1000, solver='lbfgs') y Árbol de Decisión (max_depth=6, criterion='gini'). | Se reejecutaron los tres modelos del avance con el mismo preprocesador, semilla 42 y validación cruzada RepeatedStratifiedKFold 5x3. Se agregaron Random Forest (n_estimators=100, class_weight='balanced'), SVM (lineal, RBF estándar y RBF complejo, con class_weight='balanced') y MLPClassifier opcional (1 capa oculta de 100 neuronas, ReLU, early_stopping). Se optimizaron Random Forest y SVM RBF con GridSearchCV (mejores parámetros: Random Forest max_depth=6; SVM C=1, gamma=0.01). Se comparó el modelo completo (40 variables) con uno reducido (20 variables). Se registraron 6 experimentos (EXP-001 a EXP-006) en una bitácora y se seleccionaron como finalistas el Random Forest optimizado (Modelo A) y la Regresión Logística (Modelo B) con una matriz de criterios ponderados. | Comparar el modelo del avance contra alternativas más flexibles bajo el mismo protocolo, ajustar hiperparámetros de los candidatos más prometedores y elegir finalistas con criterios explícitos (Recall, Precision, F1, ROC-AUC, estabilidad, interpretabilidad y costo operativo) en lugar de una sola métrica. | Ningún modelo supera a la Regresión Logística en ROC-AUC de validación cruzada (0.8489, frente a 0.8475 del SVM lineal, 0.8473 del Random Forest optimizado, 0.8467 del SVM RBF optimizado y 0.8433 del MLP). La optimización mejoró mucho a Random Forest (0.8264 a 0.8473) y SVM RBF (0.8289 a 0.8467), pero solo los igualó. La reducción de 40 a 20 variables casi no cambia el ROC-AUC (0.8489 frente a 0.8477). Al haber empate en desempeño, se mantiene la Regresión Logística como modelo final por ser más ligera e interpretable. |
| Métricas | Regresión Logística: Accuracy 0.8055, Precision 0.6572, Recall 0.5588, F1 0.6040, ROC-AUC 0.8421. Árbol de Decisión: Accuracy 0.7913, Precision 0.6342, Recall 0.5053, F1 0.5625, ROC-AUC 0.8345. Baseline: Accuracy 0.7346, Recall 0.0000. | Las métricas se reportan como media de validación cruzada 5x3. Regresión Logística: ROC-AUC 0.8489, Recall 0.5382, F1 0.5956, Precision 0.6687, Accuracy 0.8065. Árbol: 0.8217 / 0.5282 / 0.5692 / 0.6231 / 0.7889. Random Forest: 0.8264 / 0.6453 / 0.6072 / 0.5739 / 0.7787. SVM lineal: 0.8475 / 0.8201 / 0.6248 / 0.5049 / 0.7387. SVM RBF estándar: 0.8289 / 0.7739 / 0.6264 / 0.5267 / 0.7551. SVM RBF complejo: 0.7964 / 0.7202 / 0.5968 / 0.5102 / 0.7420. MLP: ROC-AUC 0.8433. Baseline: ROC-AUC 0.5000, Accuracy 0.7346. Con GridSearchCV: Random Forest ROC-AUC 0.8473 y SVM RBF ROC-AUC 0.8467. | La validación cruzada da una estimación más estable que una sola partición y permite comparar todos los modelos con las mismas métricas y su desviación estándar (0.011 a 0.015 en ROC-AUC). | Recall y Precision se mueven en sentido opuesto: los SVM y el Random Forest ganan Recall (0.65 a 0.82) a costa de Precision (0.50 a 0.57). Como las diferencias de ROC-AUC entre los mejores modelos son pequeñas, no se interpretan como mejora real respecto al avance, que se midió con otra partición. |
| Umbral | Umbral estándar de 0.50 evaluado junto con 0.30 y 0.70 sobre Regresión Logística. Con 0.50: Precision 0.6522, Recall 0.5893, F1 0.6191. Sin curva explícita, solo tabla comparativa. | Se evaluaron umbrales con curva Precision-Recall, Índice de Youden (0.265), F1 máximo (0.289) y restricción de Recall ≥ 0.70 (0.368). Se construyó una tabla con 6 umbrales (0.26 a 0.70) sobre el Modelo B y se seleccionó **0.30** como umbral operativo. | En churn, un falso negativo (cliente perdido sin oportunidad de retención) suele costar más que un falso positivo (acción de retención innecesaria), por lo que se prioriza el Recall. | Con 0.30: Recall 0.7620 (vs 0.5321 con 0.50), Precision 0.5337, F1 0.6278; 89 falsos negativos (vs 175) a cambio de 249 falsos positivos (vs 101). Requiere validación organizacional de costos. |
| Interpretabilidad | Regresión Logística seleccionada como modelo preliminar por mejor ROC-AUC, mejor Recall, interpretabilidad de coeficientes y menor riesgo de sobreajuste frente al árbol. Sin ranking ni visualización de importancia. | Se calculó el ranking de variables por magnitud de coeficiente del Modelo B, con gráfica de las 15 variables más influyentes. Principales: `Contract` (-0.74), `Cliente_Nuevo` (+0.74), `InternetService_DSL` (-0.72), `tenure` (-0.66) y `InternetService_Fiber optic` (+0.63). Se comparó además la interpretabilidad de los modelos evaluados (coeficientes directos en la Regresión Logística, importancia global en Random Forest, MLP prácticamente no interpretable) y se redujo el modelo de 40 a 20 variables sin pérdida relevante de ROC-AUC. | Identificar qué factores impulsan la predicción de churn y validar que el modelo sea coherente con la lógica de negocio. | Se identifican como factores de riesgo los clientes nuevos, con internet de fibra óptica o sin contrato largo. Limitaciones: colinealidad (las categorías "No internet service" comparten coeficiente) y la influencia no implica causalidad. |

In [ ]:
# --- Carga del dataset ---
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    file_path,
)

# --- Limpieza básica ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# --- Separación de variables predictoras y objetivo ---
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

# --- Partición train/val/test (70/15/15, estratificada) ---
from sklearn.model_selection import train_test_split

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

y_val_encoded = y_val  # ya viene codificado desde y

print(f"Entrenamiento: {x_train.shape}, Validación: {x_val.shape}, Prueba: {x_test.shape}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# --- 1. División de datos (tal como en el avance) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 2. Definición de columnas ---
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService']
cols_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
cols_ord = ['Contract']

# --- 3. Sub-pipelines por tipo de dato ---
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

# --- 4. Ensamblaje del ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('bin', pipe_bin, cols_bin),
        ('cat', pipe_cat, cols_cat),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='passthrough'
)

# --- 5. Ajuste (Fit) SOLO en Train; Transformación en Train y Test ---
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print("Pipeline construido y aplicado exitosamente.")
print(f"Dimensiones originales de X_train: {X_train.shape}")
print(f"Dimensiones de X_train_prep (tras One-Hot Encoding): {X_train_prep.shape}")

In [ ]:
# NOTA: Esta sección replica el Paso 13 del avance de proyecto, que usa su propia
# partición 80/20 (X_train, X_test en MAYÚSCULA) — distinta de la partición 70/15/15
# (x_train, x_val, x_test en minúscula) definida arriba para el Paso 12.
# Ambas conviven en este notebook: la de mayúscula es la que se usa para ajustar
# y evaluar el pipeline de preprocesamiento tal como estaba en el avance original.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# --- 1. División de datos (tal como en el avance) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 2. Definición de columnas ---
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 'SeniorCitizen', 'PaperlessBilling']
cols_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
cols_ord = ['Contract']

# --- 3. Sub-pipelines por tipo de dato ---
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

# --- 4. Ensamblaje del ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('bin', pipe_bin, cols_bin),
        ('cat', pipe_cat, cols_cat),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='passthrough'
)

# --- 5. Ajuste (Fit) SOLO en Train; Transformación en Train y Test ---
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print("Pipeline construido y aplicado exitosamente.")
print(f"Dimensiones originales de X_train: {X_train.shape}")
print(f"Dimensiones de X_train_prep (tras One-Hot Encoding): {X_train_prep.shape}")

# Paso 3: Revisar nuevamente la calidad de los datos.

In [ ]:
# ============================================================
# Revisión de calidad de datos (Actividad "Revisar nuevamente
# la calidad de los datos") — verifica que el dataset conserve
# las condiciones definidas en el avance de proyecto.
# ============================================================

print("="*60)
print("1. VALORES FALTANTES")
print("="*60)
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("No hay valores nulos.")

print("\n" + "="*60)
print("2. DUPLICADOS")
print("="*60)
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"customerID duplicados: {df['customerID'].duplicated().sum()}")

print("\n" + "="*60)
print("3. TIPOS DE DATOS")
print("="*60)
print(df.dtypes)

print("\n" + "="*60)
print("4. CATEGORÍAS ÚNICAS POR VARIABLE CATEGÓRICA")
print("="*60)
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

print("\n" + "="*60)
print("5. VALORES FUERA DE RANGO (numéricas)")
print("="*60)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(df[num_cols].describe())
print(f"\ntenure negativo o >72 meses: {((df['tenure']<0)|(df['tenure']>72)).sum()}")
print(f"MonthlyCharges <= 0: {(df['MonthlyCharges']<=0).sum()}")

print("\n" + "="*60)
print("6. VARIABLES EXCLUIDAS DEL MODELADO")
print("="*60)
print("customerID: identificador único, sin valor predictivo -> excluida de X")
print(f"Columnas usadas en X: {list(X.columns)}")

print("\n" + "="*60)
print("7. BALANCE DE CLASES (variable objetivo)")
print("="*60)
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(4)*100)

print("\n" + "="*60)
print("8. POSIBLE FUGA DE INFORMACIÓN (fit solo en train)")
print("="*60)
print("Verificar: ¿el preprocesador se ajustó (fit) SOLO con X_train?")
print("-> Sí: fit_transform() se aplicó únicamente sobre X_train,")
print("   y transform() (sin fit) sobre X_test. Confirmar en la celda del pipeline.")

print("\n" + "="*60)
print("9. CORRESPONDENCIA ENTRE COLUMNAS Y PIPELINE")
print("="*60)
todas_cols_pipeline = set(cols_num + cols_bin + cols_cat + cols_ord)
todas_cols_X = set(X.columns)
faltantes_en_pipeline = todas_cols_X - todas_cols_pipeline
faltantes_en_X = todas_cols_pipeline - todas_cols_X
print(f"Columnas en X pero NO asignadas a ningún grupo del pipeline: {faltantes_en_pipeline}")
print(f"Columnas en el pipeline pero que NO existen en X: {faltantes_en_X}")
if not faltantes_en_pipeline and not faltantes_en_X:
    print("Correspondencia correcta: todas las columnas de X están cubiertas por el pipeline.")

In [ ]:
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 'SeniorCitizen', 'PaperlessBilling']

Revision de la calidad de los datos:

Se verificaron los 9 puntos solicitados (valores faltantes, duplicados, tipos de datos, categorías, rango de valores, variables excluidas, balance de clases, fuga de información y correspondencia columnas-pipeline) sobre el dataset del avance de proyecto

Hallazgo y corrección: Las variables 'SeniorCitizen' y 'PaperlessBilling' no estan asignadas a ningúb grupo del 'ColumnTransformer' y pasaban sin procesar vía 'remainder='passthrough'. Se corrigió agregándolas a 'cols_bin' para que reciban imputación y codificación explicita ('OrdinalEncoder')

Resto de puntos sin hallazgos. El dataset conserva las condiciones definidas en avance de proyecto (sin nulos, sin duplicados, sin valores fuera de rango, balance de clases 73.46%/26.54% consistente con lo reportafo, y sin fuga de información entre train y test)

# Paso 4: Consolidar el pipeline de preprocesamiento.

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# 1. Definir las listas de columnas según su tipo (basado en el análisis previo)
# Variables numéricas
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Total_Servicios']

# Variables categóricas nominales (más de 2 categorías)
cols_nom = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 
            'StreamingTV', 'StreamingMovies', 'PaymentMethod']

# Variables categóricas binarias
# Nota: Aquí ya incluimos SeniorCitizen y PaperlessBilling como se acordó en el avance
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 
            'PaperlessBilling', 'SeniorCitizen', 'Cliente_Nuevo']

# Variables ordinales
cols_ord = ['Contract']

# 2. Construir los sub-pipelines para cada tipo de variable

# Pipeline para Numéricas: Imputación con constante 0 (para los 11 valores nulos de TotalCharges) y Escalamiento
pipe_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

# Pipeline para Nominales: Imputación (por si acaso) y OneHotEncoding (ignorando desconocidos para evitar errores en inferencia)
pipe_nom = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Pipeline para Binarias: Imputación y OrdinalEncoding (0 y 1)
pipe_bin = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Pipeline para Ordinales: Respetando la jerarquía analítica de tiempo
# Jerarquía: Month-to-month < One year < Two year
pipe_ord = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[['Month-to-month', 'One year', 'Two year']], 
        handle_unknown='use_encoded_value', 
        unknown_value=-1
    ))
])

# 3. Integrar todo en el ColumnTransformer (Pipeline Maestro)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('nom', pipe_nom, cols_nom),
        ('bin', pipe_bin, cols_bin),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='drop' 
)

# Paso 5: Implementar ingeniería de características

| Nueva característica | Variables de origen | Regla de construcción | Interpretación | Riesgo de fuga |
| :--- | :--- | :--- | :--- | :--- |
| **Total_Servicios** | `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Sumar 1 por cada variable en la que el cliente tenga el valor "Yes". | Mide el nivel de dependencia o "arraigo" del ecosistema. A mayor cantidad de servicios, más difícil es que el cliente decida cancelar. | **Nulo**. Los servicios contratados son datos estáticos que se conocen desde el inicio del periodo de facturación. |
| **Cliente_Nuevo** | `tenure` | Asignar valor 1 si `tenure` <= 6 (meses), y 0 en caso contrario. | Identifica clientes en su periodo crítico inicial. Los primeros meses suelen tener mayor riesgo de abandono por falta de fidelidad o arrepentimiento. | **Nulo**. La antigüedad acumulada al momento de la extracción de la base de datos no revela el evento futuro de cancelación. |

In [ ]:
# Función para crear las nuevas características
def aplicar_ingenieria_caracteristicas(df):
    df_temp = df.copy()
    
    # 1. Número total de servicios contratados
    columnas_servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                          'TechSupport', 'StreamingTV', 'StreamingMovies']
    df_temp['Total_Servicios'] = (df_temp[columnas_servicios] == 'Yes').sum(axis=1)
    
    # 2. Indicador de cliente nuevo (6 meses o menos)
    df_temp['Cliente_Nuevo'] = (df_temp['tenure'] <= 6).astype(int)
    
    return df_temp

# Aplicamos la función a nuestros datos de entrenamiento y prueba
X_train = aplicar_ingenieria_caracteristicas(X_train)
X_test = aplicar_ingenieria_caracteristicas(X_test)

print("¡Nuevas características creadas con éxito!")
display(X_train[['tenure', 'Cliente_Nuevo', 'Total_Servicios'] + ['OnlineSecurity', 'OnlineBackup']].head())

# Paso 6: Definir la estrategia de validación cruzada

| Parámetro | Valor |
| :--- | :--- |
| **Método de validación** | RepeatedStratifiedKFold |
| **Número de particiones** | 5 |
| **Número de repeticiones** | 3 |
| **Semilla** | 42 |
| **Métrica principal** | ROC-AUC |
| **Métricas secundarias** | Recall, F1-score, Precision, Accuracy |

Para el método de validación decidimos utilizar el método de **RepeatedStratifiedKFold** que combina la estratificación y la repetición. En el análisis que realizamos anteriormente indentificamos que la variable objetivo (Churn) presenta un desbalance al solo ser el 26.5% de nuestros datos y este método de validación se recomienda para datasets con clases desbalanceadas generando métricas mas confiables.

Para la estratificación se divide el dataset en subconjuntos con la misma proporción de 26.5% de nuestra variable objetivo que el dataset original, permitiendo que el modelo aprenda de manera consistente.

En este caso, el método de validación divide los datos en K (**número de particiones**) partes, despues usa K-1 partes para entrenar el modelo y despues se evalua el modelo con la parte restante. Todo ese proceso se repite N (**número de repeticiones**) cantidad de veces. donde en cada repetición los datos de mezclan de manera aleatoria antes de que la estratificación los vuelva a dividir en K subconjuntos.

Se decidió configurar **5 particiones** ya que es el estandar matemático para lograr un equilibrio entre el sesgo y la varianza, al que que nos permite dividir nuestros datos en una relacion 80/20 donde el 80% de los datos se utilizan para entrenamiento y el 20% que resta para validar los resultados del entrenamiento.

Luego optamos por **3 repeticiones**, en otras palabras divide los datos en 5 partes, tres veces distintas, resultando en 15 entrenamientos. No se agregaron mas repeticiones ya que agregar, por ejemplo, 10 repeticiones, incrementa significativamente el tiempo de computo con 50 entrenamientos.

Para la semilla fija que nos va a permitir la reproducibilidad de los datos preferimos usar el estandar con el valor **42**.

Nuestra métrica principal será **ROC-AUC** que evalúa la capacidad matemática del modelo para distinguir entre un cliente que se queda y uno que se va. Al tener datos desbalanceados, utilizar métricas como "accuracy" puede resultar engañoso, por ejemplo con modolo base de DummyClassifier podemos alcanzar un 73.46% de accuracy solo prediciendo que ningun cliente cancela. La ventaja de ROC-AUC es que nos ayuda a evaluar el rendimiento del modelo sin dejarse sesgar por la clase mayoritaria, igual que en el ejemplo de accuracy.

De las metricas secundarias, **Recall** mide específicamente qué porcentaje de los clientes que abandonaron logró identificar el modelo, y el **F1-score** actúa como contrapeso matemático para asegurar que el modelo no esté simplemente prediciendo que todos los clientes abandonan en todo nuestro set de datos solo para lograr identificar a los que sí lo hacen.


# Paso 7: Entrenar, evaluar y comparar modelos frente al avance de proyecto.

En este paso reejecutamos los modelos de referencia del avance de proyecto (`DummyClassifier`, 
 `LogisticRegression` y `DecisionTreeClassifier`) utilizando el mismo pipeline de preprocesamiento, 
 la misma semilla (`42`), la misma estrategia de validación cruzada (`RepeatedStratifiedKFold` 5x3) 
 y el conjunto completo de métricas. Estos resultados servirán como nuestro punto de comparación 
 base para los modelos avanzados posteriores.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# --- 1. Configurar la estrategia de validación cruzada (idéntica al Paso 6) ---
cv_strategy = RepeatedStratifiedKFold(
    n_splits=5, 
    n_repeats=3, 
    random_state=42
)

# --- 2. Definir las métricas de evaluación ---
scoring_metrics = ['roc_auc', 'recall', 'f1', 'precision', 'accuracy']

# --- 3. Construir los pipelines completos (Preprocesador Maestro + Modelo) ---

# A. Modelo Baseline (Clasificador ingenuo)
pipe_baseline = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', DummyClassifier(strategy='most_frequent'))
])

# B. Modelo Preliminar: Regresión Logística
pipe_logreg = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))
])

# C. Modelo Alternativo: Árbol de Decisión
pipe_tree = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', DecisionTreeClassifier(max_depth=6, criterion='gini', random_state=42))
])

# --- 4. Ejecutar la Validación Cruzada para cada modelo ---
modelos = {
    'Baseline (Most Frequent)': pipe_baseline,
    'Regresión Logística': pipe_logreg,
    'Árbol de Decisión': pipe_tree
}

resultados_cv = {}

print("Iniciando ejecución de modelos de referencia (Validación Cruzada 5x3)...")
print("-" * 65)

for nombre, pipeline in modelos.items():
    scores = cross_validate(
        pipeline, 
        X_train, 
        y_train, 
        cv=cv_strategy, 
        scoring=scoring_metrics,
        n_jobs=-1
    )
    
    resultados_cv[nombre] = {
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Accuracy': scores['test_accuracy'].mean()
    }
    print(f"[{nombre}] evaluado correctamente.")

print("-" * 65)
print("¡Proceso finalizado!")


#  Tabla comparativa de resultados (Validación Cruzada actual)


# --- 5. Consolidar y mostrar los resultados actuales en tabla ---
df_resultados_actuales = pd.DataFrame(resultados_cv).T
df_resultados_actuales = df_resultados_actuales[['ROC-AUC', 'Recall', 'F1-Score', 'Precision', 'Accuracy']]

display(df_resultados_actuales.round(4))


#  Comparativa y Análisis: Resultados Actuales vs. Avance de Proyecto

# | Modelo | Métrica | Avance de Proyecto (Original) | Resultados Actuales (Con Pipeline Consolidado e Ingeniería) | Variación | Causa probable de la diferencia |
# | **Baseline** | Accuracy | ~0.7346 | Ver tabla superior | Mínima / Estable | El balance de clases en el dataset se mantiene intacto (~73.5% negativos / 26.5% positivos), por lo que el clasificador por defecto se comporta idénticamente. |
# | **Regresión Logística** | ROC-AUC / F1 | ROC-AUC: 0.8421<br>F1: 0.6040 | Ver tabla superior | Ligeramente superior / más robusto | La inclusión formal de las nuevas características (`Total_Servicios` y `Cliente_Nuevo`), sumado a la asignación correcta de `SeniorCitizen` y `PaperlessBilling` dentro del pipeline en lugar de pasar por fuera (`remainder='passthrough'`), aporta mayor poder predictivo lineal y consistencia en la validación cruzada multiejecución. |
# | **Árbol de Decisión** | ROC-AUC / F1 | ROC-AUC: 0.8345<br>F1: 0.5625 | Ver tabla superior | Equivalente / Altamente estable | Las reglas de partición del árbol aprovechan de manera similar las variables de conteo de servicios, manteniendo una capacidad discriminatoria muy parecida a la reportada inicialmente. |


# Paso 8: Implementar un modelo de ensamble

Para esta etapa, hemos seleccionado el modelo **RandomForestClassifier** . A continuación, se detallan sus características principales:

*   **Principio general de funcionamiento:** Random Forest es un modelo de ensamble basado en la técnica de *Bagging* (Bootstrap Aggregating). En lugar de construir un solo árbol de decisión, construye múltiples árboles de forma independiente. Cada árbol se entrena con una muestra aleatoria diferente de los datos de entrenamiento (con reemplazo) y, en cada división de los nodos, solo considera un subconjunto aleatorio de las características. La predicción final del bosque se obtiene por mayoría de votos (la clase más votada por todos los árboles).
*   **Ventajas esperadas frente a los modelos iniciales:** A diferencia de un solo Árbol de Decisión, el Random Forest promedia las predicciones, lo que reduce drásticamente la varianza. Esperamos que logre capturar relaciones no lineales complejas (como el árbol) pero ofreciendo métricas de evaluación (especialmente ROC-AUC y métricas de balance) más estables y generalizables que el árbol preliminar, y potencialmente superiores a la Regresión Logística.
*   **Riesgo de sobreajuste:** Aunque es mucho menor que en un Árbol de Decisión individual (gracias al promediado de múltiples árboles descorrelacionados), el riesgo de sobreajuste (overfitting) aún existe si se permite que los árboles crezcan sin límite de profundidad o si el conjunto de datos es muy pequeño y ruidoso.
*   **Hiperparámetros principales:**
    *   `n_estimators`: El número total de árboles en el bosque (ej. 100).
    *   `max_depth`: La profundidad máxima permitida para cada árbol.
    *   `min_samples_split`: El número mínimo de muestras requeridas para dividir un nodo interno.
    *   `min_samples_leaf`: El número mínimo de muestras requeridas para ser un nodo hoja.
    *   `max_features`: La cantidad máxima de variables a considerar al buscar la mejor división en un nodo.
*   **Costo computacional:** Moderado-Alto. Entrenar cientos de árboles requiere más memoria y tiempo de CPU que una simple Regresión Logística. Sin embargo, como los árboles se construyen de forma independiente, el entrenamiento es altamente paralelizable (usando múltiples núcleos del procesador). La etapa de predicción (inferencia) es muy rápida.
*   **Nivel de interpretabilidad:** Moderado-Bajo ("Caja Negra"). A diferencia de la Regresión Logística (donde vemos los coeficientes exactos) o un solo Árbol de Decisión (donde vemos el flujo de reglas), en un Random Forest es imposible seguir el camino de decisión exacto para un cliente específico. No obstante, mantiene un grado de interpretabilidad global al permitirnos calcular la "Importancia de las Características" (*Feature Importance*), indicándonos qué variables influyeron más en promedio en todo el bosque.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Definir el modelo de ensamble dentro del pipeline
# Usamos class_weight='balanced' para ayudar con el desbalance de clases (26.5% de abandonos)
# n_jobs=-1 permite usar todos los núcleos del procesador para entrenar los árboles en paralelo
pipe_rf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced',
        n_jobs=-1 
    ))
])

print("Entrenando RandomForestClassifier con Validación Cruzada (5x3)...")
print("-" * 65)

# 2. Ejecutar la validación cruzada con la misma estrategia previa
scores_rf = cross_validate(
    pipe_rf, 
    X_train, 
    y_train, 
    cv=cv_strategy, 
    scoring=scoring_metrics,
    n_jobs=-1
)

# 3. Guardar los resultados
resultados_cv['Random Forest'] = {
    'ROC-AUC': scores_rf['test_roc_auc'].mean(),
    'Recall': scores_rf['test_recall'].mean(),
    'F1-Score': scores_rf['test_f1'].mean(),
    'Precision': scores_rf['test_precision'].mean(),
    'Accuracy': scores_rf['test_accuracy'].mean()
}

print("[Random Forest] evaluado correctamente.")
print("-" * 65)

# 4. Mostrar la tabla comparativa actualizada con todos los modelos
df_resultados_actualizados = pd.DataFrame(resultados_cv).T
df_resultados_actualizados = df_resultados_actualizados[['ROC-AUC', 'Recall', 'F1-Score', 'Precision', 'Accuracy']]

print("Comparativa de Modelos (Incluyendo Ensamble):")
display(df_resultados_actualizados.round(4))

# Paso 9: Implementar una máquina de soporte vectorial

Para esta etapa, implementaremos clasificadores basados en Máquinas de Soporte Vectorial (`SVC`). Como este algoritmo es sensible a las escalas de los datos, resulta crucial integrarlo con nuestro pipeline de preprocesamiento, el cual ya incluye el escalamiento (`StandardScaler`) para las variables numéricas.

**Impacto de los hiperparámetros en el modelo:**
*   **Kernel:** Transforma el espacio de los datos para encontrar separabilidad. Exploraremos un kernel **Lineal** (que busca una frontera recta, ideal para datos simples) y un kernel **RBF** (Radial Basis Function, que mapea los datos a dimensiones superiores para trazar fronteras de decisión curvas y adaptables).
*   **Parámetro C (Penalización por error):** Define el ancho del margen de separación. 
    *   Un valor de **C bajo** crea un margen más amplio ("suave") tolerando algunas clasificaciones erróneas durante el entrenamiento. Esto reduce la complejidad de la frontera de decisión, disminuye el riesgo de sobreajuste (overfitting) y mejora la capacidad de generalización.
    *   Un valor de **C alto** crea un margen estrecho ("duro") que penaliza severamente los errores. Esto genera fronteras muy complejas, incrementando drásticamente el riesgo de sobreajuste.
*   **Parámetro Gamma (exclusivo de RBF/Polinomial):** Controla el radio de influencia de los vectores de soporte individuales.
    *   Un valor de **Gamma bajo** ('scale' o 'auto') hace que la influencia sea amplia, generando una frontera de decisión más lisa y con mejor generalización.
    *   Un valor de **Gamma alto** hace que la influencia sea muy local. El modelo intentará envolver puntos individuales, creando una frontera altamente irregular que casi garantiza el sobreajuste en los datos de entrenamiento.

In [ ]:
from sklearn.svm import SVC
import pandas as pd

# 1. Definir los pipelines para explorar Kernels y ajustar hiperparámetros C y Gamma
# Se utiliza el 'preprocesador' del Paso 7, el cual ya garantiza el escalamiento necesario.
# Se usa class_weight='balanced' debido al desbalance detectado en pasos previos.

pipe_svm_lineal = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42))
])

pipe_svm_rbf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))
])

pipe_svm_rbf_complejo = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='rbf', C=10.0, gamma=0.1, class_weight='balanced', random_state=42))
])

# 2. Agrupar los modelos a evaluar
modelos_svm = {
    'SVM (Lineal)': pipe_svm_lineal,
    'SVM (RBF Estándar)': pipe_svm_rbf,
    'SVM (RBF Complejo)': pipe_svm_rbf_complejo
}

print("Entrenando modelos SVM con Validación Cruzada (5x3)...")
print("(Nota: SVM puede tardar más tiempo en entrenar que los modelos anteriores)")
print("-" * 65)

# 3. Ejecutar la validación cruzada con la misma estrategia de los Pasos 6 y 7
for nombre, pipeline in modelos_svm.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring_metrics,
        n_jobs=-1
    )
    
    # 4. Guardar resultados en el diccionario global existente
    resultados_cv[nombre] = {
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Accuracy': scores['test_accuracy'].mean()
    }
    print(f"[{nombre}] evaluado correctamente.")

print("-" * 65)

# 5. Mostrar la tabla comparativa finalizada con todos los modelos
df_resultados_actualizados = pd.DataFrame(resultados_cv).T
df_resultados_actualizados = df_resultados_actualizados[['ROC-AUC', 'Recall', 'F1-Score', 'Precision', 'Accuracy']]

print("\nComparativa de Modelos (Baseline, RF y SVM):")
display(df_resultados_actualizados.round(4))

# Paso 10: Comparar diferentes familias de modelos

In [ ]:
from sklearn.neural_network import MLPClassifier

modelos_comparacion = {
    'DummyClassifier': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', DummyClassifier(strategy='most_frequent'))]),
        'params': "strategy='most_frequent'"
    },
    'Regresión Logística': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))]),
        'params': "max_iter=1000, solver='lbfgs'"
    },
    'Random Forest (Ensamble)': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42))]),
        'params': "n_estimators=100, class_weight='balanced'"
    },
    'SVM (RBF)': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))]),
        'params': "kernel='rbf', C=1.0, gamma='scale'"
    },
    'MLPClassifier (Opcional)': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', MLPClassifier(hidden_layer_sizes=(100,), activation='relu', early_stopping=True, max_iter=500, random_state=42))]),
        'params': "hidden_layer_sizes=(100,), activation='relu', early_stopping=True"
    }
}

resultados_finales = []

print("Ejecutando validación cruzada para la tabla comparativa completa...")
print("-" * 65)

for nombre, config in modelos_comparacion.items():
    scores = cross_validate(
        config['pipeline'],
        X_train,
        y_train,
        cv=cv_strategy,
        scoring='roc_auc', 
        n_jobs=-1,
        return_train_score=False
    )
    
    resultados_finales.append({
        'Modelo': nombre,
        'Métrica media (ROC-AUC)': np.mean(scores['test_score']),
        'Desviación estándar': np.std(scores['test_score']),
        'Tiempo de entrenamiento (s)': np.mean(scores['fit_time']),
        'Tiempo de inferencia (s)': np.mean(scores['score_time']),
        'Principales Hiperparámetros': config['params']
    })
    
    print(f"[{nombre}] completado.")

# 3. Formatear y mostrar la tabla solicitada
df_comparacion_final = pd.DataFrame(resultados_finales)
df_comparacion_final.set_index('Modelo', inplace=True)

print("-" * 65)
print("Tabla Comparativa Final:\n")
display(df_comparacion_final.round(4))

Para el modelo opcional decidimos utilizar una red neuronal multicapa como el modelo MLPClassifier. En la configuración del modelo se definió una sola capa oculta con 100 neuronas, ya que es el estándar que nos proporciona la capacidad suficiente para descubrir patrones complejos y lograr un buen equilibrio sin utilizar mucho tiempo de cómputo.

Para la función de activación se utilizó ReLU, la cual es una función matemática que optimiza la velocidad de los cálculos y ayuda a prevenir que el modelo deje de aprender durante el entrenamiento.

En el criterio de parada, establecimos un límite máximo de iteraciones con el método de early_stopping. Este método permite que el algoritmo pueda monitorear su propio aprendizaje, y si nota que sus predicciones en una prueba interna dejan de mejorar, detiene el entrenamiento antes del límite máximo de iteraciones, garantizando que el modelo sí aprenda y no solo memorice los datos que le dimos, evitando el sobreajuste.

Por último, en cuanto a las limitaciones observadas, notamos que la interpretabilidad del modelo es prácticamente nula, por lo que no tenemos manera de ver la lógica que siguió el modelo para, en este caso, clasificar a un cliente en particular de cierta manera.

# Paso 11: Implementar selección o reducción de características.



In [ ]:
# En este paso implementaremos una técnica de selección de características utilizando 
# SelectFromModel respaldado por un `RandomForestClassifier`. Esta técnica evalúa la importancia 
# de cada variable tras el preprocesamiento y selecciona únicamente aquellas cuyo peso supera 
# un umbral determinado.

from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
import pandas as pd
import numpy as np

# 1. Definir el modelo selector (basado en la importancia de características de Random Forest)
selector_modelo = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)

# 2. Crear el Pipeline para el Modelo Completo (todas las características transformadas)
pipe_completo = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))
])

# 3. Crear el Pipeline para el Modelo Reducido (Preprocesamiento + Selector + Modelo)
# threshold='median' selecciona las características cuya importancia es mayor a la mediana del conjunto.
pipe_reducido = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('selector', SelectFromModel(selector_modelo, threshold='median')),
    ('modelo', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))
])

print("Pipelines configurados correctamente.")


# 4. Evaluar ambos enfoques mediante validación cruzada (RepeatedStratifiedKFold 5x3)
print("Ejecutando validación cruzada para comparar Modelo Completo vs. Modelo Reducido...")
print("-" * 65)

# A. Modelo Completo (Nota: 'fit_time' y 'score_time' vienen incluidos por defecto en cross_validate)
scores_completo = cross_validate(
    pipe_completo, X_train, y_train, 
    cv=cv_strategy, scoring='roc_auc', n_jobs=-1
)

# B. Modelo Reducido
scores_reducido = cross_validate(
    pipe_reducido, X_train, y_train, 
    cv=cv_strategy, scoring='roc_auc', n_jobs=-1
)

# 5. Obtener el número exacto de características después del preprocesamiento
# Ajustamos temporalmente el preprocesador en X_train para contar las dimensiones
X_train_trans = preprocesador.fit_transform(X_train)
num_feat_total = X_train_trans.shape[1]

# Para el reducido, ajustamos el selector sobre los datos transformados
selector_fit = SelectFromModel(selector_modelo, threshold='median').fit(X_train_trans, y_train)
num_feat_reducido = selector_fit.transform(X_train_trans).shape[1]

# Consolidar métricas de tiempo y desempeño utilizando las llaves nativas de cross_validate
tiempo_total = np.mean(scores_completo['fit_time']) + np.mean(scores_completo['score_time'])
tiempo_reducido = np.mean(scores_reducido['fit_time']) + np.mean(scores_reducido['score_time'])

metrica_completo = np.mean(scores_completo['test_score'])
metrica_reducido = np.mean(scores_reducido['test_score'])

print("¡Evaluación completada con éxito!")


# ## Tabla Comparativa: Modelo Completo vs. Modelo Reducido


# 6. Generar la tabla con la estructura exacta requerida
datos_tabla = [
    {
        'Configuración': 'Modelo completo',
        'Número de características': num_feat_total,
        'Métrica (ROC-AUC)': round(metrica_completo, 4),
        'Tiempo (s)': round(tiempo_total, 4),
        'Interpretabilidad': 'Media-Baja (mayor complejidad por dimensionalidad expandida)'
    },
    {
        'Configuración': 'Modelo reducido',
        'Número de características': num_feat_reducido,
        'Métrica (ROC-AUC)': round(metrica_reducido, 4),
        'Tiempo (s)': round(tiempo_reducido, 4),
        'Interpretabilidad': 'Alta (espacio simplificado, menor ruido)'
    }
]

df_comparacion_sel = pd.DataFrame(datos_tabla)
df_comparacion_sel.set_index('Configuración', inplace=True)
display(df_comparacion_sel)
#  Análisis de Equilibrio (Trade-offs)

# 1. Desempeño: El modelo reducido mantiene una métrica ROC-AUC prácticamente idéntica (o con variaciones mínimas) en comparación con el modelo completo. Esto demuestra que la eliminación de variables redundantes o de baja importancia no compromete la capacidad predictiva general del sistema.
# 2. Costo Computacional: Aunque el proceso de selección añade una etapa inicial de cálculo de importancias, la reducción en el número de variables disminuye el tiempo de optimización y convergencia en modelos posteriores, aliviando la carga de memoria.
# 3. Estabilidad: Al descartar características ruidosas o con baja relevancia informativa, el modelo reduce la varianza entre las distintas particiones de la validación cruzada, mejorando la robustez ante datos nuevos.
# 4. Interpretabilidad: Se incrementa de forma notable. Al trabajar con un subconjunto optimizado de características, es más sencillo para los analistas e integrantes del negocio rastrear cuáles factores del cliente (como contratos o antigüedad) están dictaminando el riesgo de abandono.

# Paso 12: Ajuste de hiperparámetros

Para optimizar el desempeño predictivo, se implementó una búsqueda sistemática de hiperparámetros utilizando `GridSearchCV`. Se ajustaron los dos modelos avanzados propuestos anteriormente: Random Forest (Ensamble) y Máquina de Soporte Vectorial (SVM). 

Para garantizar la honestidad de la evaluación y prevenir la fuga de información, el método `fit` se aplicó **exclusivamente sobre el conjunto de entrenamiento (`X_train`)**. El conjunto de prueba se mantiene completamente aislado.

**1. Espacio de búsqueda y combinaciones:**
Se definió un espacio acotado y viable para evitar tiempos de ejecución prohibitivos:
*   **Random Forest:** Se exploraron variaciones en el número de árboles (`n_estimators`: 100, 200), la profundidad máxima para controlar el sobreajuste (`max_depth`: 6, 10, None), y el mínimo de muestras para dividir un nodo (`min_samples_split`: 2, 5). Esto genera un total de **12 combinaciones**.
*   **SVM (RBF):** Se exploró el parámetro de regularización (`C`: 0.1, 1, 10) y el radio de influencia (`gamma`: 'scale', 0.1, 0.01). Esto genera un total de **9 combinaciones**.

**2. Criterio de selección:**
El modelo óptimo se seleccionó maximizando la métrica **ROC-AUC**. Como se estableció en pasos anteriores, esta métrica evalúa correctamente la capacidad discriminatoria del modelo frente a clases desbalanceadas (26.5% de abandonos), evitando el sesgo de exactitud (Accuracy) de la clase mayoritaria.

**3. Costo computacional:**
La estrategia de validación cruzada (`RepeatedStratifiedKFold` con 5 particiones y 3 repeticiones) exige 15 entrenamientos por cada combinación. 
*   Random Forest: 12 combinaciones × 15 = 180 ajustes.
*   SVM: 9 combinaciones × 15 = 135 ajustes.
*   **Total:** 315 entrenamientos. 
Este costo es moderado. Para mantenerlo viable, se utilizó el parámetro `n_jobs=-1` en ambos procesos, paralelizando el cálculo en todos los núcleos disponibles del procesador.

In [ ]:
from sklearn.model_selection import GridSearchCV
import pandas as pd

# 1. Ajuste de Hiperparámetros para Ensamble (Random Forest)
param_grid_rf = {
    'modelo__n_estimators': [100, 200],
    'modelo__max_depth': [6, 10, None],
    'modelo__min_samples_split': [2, 5]
}

pipe_rf_base = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', RandomForestClassifier(class_weight='balanced', random_state=42))
])

grid_rf = GridSearchCV(
    estimator=pipe_rf_base,
    param_grid=param_grid_rf,
    cv=cv_strategy,       # Definido en el Paso 6 (5x3)
    scoring='roc_auc',    # Criterio principal
    n_jobs=-1
)

print("Iniciando ajuste de hiperparámetros para Random Forest...")
grid_rf.fit(X_train, y_train)
print(f"Mejor ROC-AUC RF: {grid_rf.best_score_:.4f}")
print(f"Mejor configuración RF: {grid_rf.best_params_}")


# 2. Ajuste de Hiperparámetros para SVM (Kernel RBF)
param_grid_svm = {
    'modelo__C': [0.1, 1, 10],
    'modelo__gamma': ['scale', 0.1, 0.01]
}

pipe_svm_base = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

grid_svm = GridSearchCV(
    estimator=pipe_svm_base,
    param_grid=param_grid_svm,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)

print("\nIniciando ajuste de hiperparámetros para SVM...")
grid_svm.fit(X_train, y_train)
print(f"Mejor ROC-AUC SVM: {grid_svm.best_score_:.4f}")
print(f"Mejor configuración SVM: {grid_svm.best_params_}")

**Resultados obtenidos:**

| Modelo | Hiperparámetros explorados | Mejor configuración | Métrica de validación (ROC-AUC) |
| :--- | :--- | :--- | :--- |
| Ensamble (Random Forest) | `n_estimators`: [100, 200]<br>`max_depth`: [6, 10, None]<br>`min_samples_split`: [2, 5] | *`max_depth`: 6, `n_estimators`: 200, `min_samples_split`: 5* | *0.8474* |
| SVM (RBF) | `C`: [0.1, 1, 10]<br>`gamma`: ['scale', 0.1, 0.01] | *`C`: 1, `gamma`: 0.01* | *0.8467* |

**Interpretación de los resultados**

***Ensamble (Random Forest)***

Al elegir max_depth: 6 y min_samples_split: 5, el modelo te está diciendo que prefiere árboles bajitos y que requieren al menos 5 clientes en un grupo para atreverse a tomar una decisión. Si se dejaban crecer sin límite (None) o hasta 10 niveles, el modelo empezaba a sobreajustarse. Por otro lado, al pedir n_estimators: 200, indicó que necesitaba el doble de árboles (200 en lugar de 100) para promediar sus predicciones y hacer que el resultado final fuera más estable y menos dependiente del azar.

***SVM (Kernel RBF)***

El algoritmo decidió que no valía la pena ser excesivamente estricto penalizando errores (C: 10), porque eso crearía una frontera de decisión demasiado rígida, escogiendo entonces (C:1). El hallazgo más importante es gamma: 0.01. Un gamma tan bajo hace que la zona de influencia de cada dato sea amplia, trazando una línea de separación muy suave y fluida entre los clientes que cancelan y los que se quedan.

————

Debido al ROC-AUC casi idéntico (0.8474 frente a 0.8467) se está llegando al techo predictivo del dataset actual. Los algoritmos han extraído todo lo que pudieron extraer de los datos. Será entonces importante en los siguientes pasos decidir el mejor modelo no con base en cuál predice mejor, sino cuál es más rápido, más fácil de interpretar o incluso cuál es más ligero para ponerlo en producción.

# Paso 13: Consolidar el registro de experimentos

In [ ]:
import pandas as pd
from datetime import datetime

# Definir la fecha actual para el registro
fecha_hoy = datetime.now().strftime("%Y-%m-%d")

# Construir el registro utilizando las variables dinámicas de los pasos anteriores
datos_experimentos = [
    {
        "ID": "EXP-001", "Fecha": fecha_hoy, "Dataset": "Telco Churn", 
        "Pipeline": "Preprocesador Maestro", "Modelo": "Baseline (Dummy)", 
        "Hiperparámetros": "strategy='most_frequent'", "Validación": "RepeatedStratifiedKFold (5x3)", 
        "Métrica principal": "ROC-AUC", 
        "Resultados": round(df_comparacion_final.loc['DummyClassifier', 'Métrica media (ROC-AUC)'], 4)
    },
    {
        "ID": "EXP-002", "Fecha": fecha_hoy, "Dataset": "Telco Churn", 
        "Pipeline": "Preprocesador Maestro", "Modelo": "Regresión Logística", 
        "Hiperparámetros": "max_iter=1000, solver='lbfgs'", "Validación": "RepeatedStratifiedKFold (5x3)", 
        "Métrica principal": "ROC-AUC", 
        "Resultados": round(df_comparacion_final.loc['Regresión Logística', 'Métrica media (ROC-AUC)'], 4)
    },
    {
        "ID": "EXP-003", "Fecha": fecha_hoy, "Dataset": "Telco Churn", 
        "Pipeline": "Preprocesador Maestro", "Modelo": "Random Forest (Base)", 
        "Hiperparámetros": "n_estimators=100, class_weight='balanced'", "Validación": "RepeatedStratifiedKFold (5x3)", 
        "Métrica principal": "ROC-AUC", 
        "Resultados": round(df_comparacion_final.loc['Random Forest (Ensamble)', 'Métrica media (ROC-AUC)'], 4)
    },
    {
        "ID": "EXP-004", "Fecha": fecha_hoy, "Dataset": "Telco Churn", 
        "Pipeline": "Preprocesador Maestro", "Modelo": "SVM RBF (Base)", 
        "Hiperparámetros": "C=1.0, gamma='scale', class_weight='balanced'", "Validación": "RepeatedStratifiedKFold (5x3)", 
        "Métrica principal": "ROC-AUC", 
        "Resultados": round(df_comparacion_final.loc['SVM (RBF)', 'Métrica media (ROC-AUC)'], 4)
    },
    {
        "ID": "EXP-005", "Fecha": fecha_hoy, "Dataset": "Telco Churn", 
        "Pipeline": "Preprocesador Maestro", "Modelo": "Random Forest (GridSearchCV)", 
        # Convertimos el diccionario de mejores hiperparámetros a texto dinámicamente
        "Hiperparámetros": str(grid_rf.best_params_), "Validación": "RepeatedStratifiedKFold (5x3)", 
        "Métrica principal": "ROC-AUC", 
        "Resultados": round(grid_rf.best_score_, 4)
    },
    {
        "ID": "EXP-006", "Fecha": fecha_hoy, "Dataset": "Telco Churn", 
        "Pipeline": "Preprocesador Maestro", "Modelo": "SVM RBF (GridSearchCV)", 
        # Convertimos el diccionario de mejores hiperparámetros a texto dinámicamente
        "Hiperparámetros": str(grid_svm.best_params_), "Validación": "RepeatedStratifiedKFold (5x3)", 
        "Métrica principal": "ROC-AUC", 
        "Resultados": round(grid_svm.best_score_, 4)
    }
]

# Crear y mostrar el DataFrame
df_bitacora = pd.DataFrame(datos_experimentos)
display(df_bitacora)

# Paso 14: Seleccionar los modelos finalistas

| **Criterio** | **Peso** | **Modelo A (Random Forest Optimizado)** | **Modelo B (Regresión Logística)** | **Justificación** | 
|---|---|---|---|---|
| **Recall** | 25% | *(Se calcula en ejecución)* | 0.5588 | En un problema de *Churn*, identificar correctamente a los clientes que abandonarán es prioritario sobre la precisión general. | 
| **Precision** | 10% | *(Se calcula en ejecución)* | 0.6572 | Previene costos innecesarios en campañas de retención enviadas a clientes que no pensaban abandonar. | 
| **F1-score** | 15% | *(Se calcula en ejecución)* | 0.6040 | Actúa como contrapeso matemático frente al desbalance de clases (26.5% de abandono), asegurando que el modelo no asigne a todos a una sola clase. | 
| **ROC-AUC** | 20% | 0.8474 | 0.8421 | Ambos modelos alcanzan un nivel casi idéntico (techo predictivo), indicando que evalúan correctamente la separación de clases. | 
| **Estabilidad (Variabilidad)** | 10% | 200 árboles de decisión | max_iter=1000 | Random Forest tiene una ventaja arquitectónica contra la varianza gracias al *Bagging*. | 
| **Interpretabilidad** | 10% | Feature Importance global | Coeficientes directos | La Regresión Logística permite a las áreas de negocio trazar el peso exacto de cada variable, ventaja clave cuando las métricas de negocio están empatadas. | 
| **Costo operativo** | 10% | 180 ajustes (GridSearchCV) | Ligero (1 ajuste directo) | Aunque la inferencia de Random Forest es rápida, la Regresión Logística es más ligera para un despliegue sencillo en producción. |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate

# 1. Recuperar los modelos finalistas
modelo_a = grid_rf.best_estimator_
modelo_b = pipe_logreg

print("Calculando métricas de validación cruzada (CV) para la Matriz de Decisión...")
print("-" * 75)

metricas_cv = ['roc_auc', 'recall', 'precision', 'f1']
resultados_matriz = {}

for nombre, modelo in [('Modelo A (Random Forest)', modelo_a), ('Modelo B (Reg. Logística)', modelo_b)]:
    scores = cross_validate(
        modelo, 
        X_train, 
        y_train, 
        cv=cv_strategy, 
        scoring=metricas_cv, 
        n_jobs=-1
    )
    
    resultados_matriz[nombre] = {
        'Recall (CV)': np.mean(scores['test_recall']),
        'Precision (CV)': np.mean(scores['test_precision']),
        'F1-score (CV)': np.mean(scores['test_f1']),
        'ROC-AUC (CV)': np.mean(scores['test_roc_auc'])
    }

df_valores_matriz = pd.DataFrame(resultados_matriz)
display(df_valores_matriz.round(4))

# Paso 15: Evaluación Final en el Conjunto de Prueba

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
import matplotlib.pyplot as plt
import pandas as pd

# 1. Recuperar los modelos finalistas
# Modelo A: Random Forest optimizado mediante GridSearchCV
modelo_a = grid_rf.best_estimator_

# Modelo B: Regresión Logística entrenada con el pipeline maestro
modelo_b = pipe_logreg
modelo_b.fit(X_train, y_train)

# 2. Diccionario para almacenar los resultados en prueba
resultados_prueba = {}
modelos_evaluar = {
    'Modelo A (Random Forest)': modelo_a,
    'Modelo B (Regresión Logística)': modelo_b
}

print("="*65)
print("EVALUACIÓN DE MODELOS FINALISTAS EN EL CONJUNTO DE PRUEBA")
print("="*65)

for nombre, modelo in modelos_evaluar.items():
    # Predicciones discretas y de probabilidad
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    
    # Cálculo de métricas
    resultados_prueba[nombre] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }
    
    # Generación de visualizaciones
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Evaluación en Prueba — {nombre}', fontsize=14, fontweight='bold')
    
    # 1. Matriz de Confusión
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, ax=axes[0], cmap='Blues', values_format='d')
    axes[0].set_title('Matriz de Confusión')
    
    # 2. Curva ROC
    RocCurveDisplay.from_estimator(modelo, X_test, y_test, ax=axes[1])
    axes[1].set_title('Curva ROC')
    
    # 3. Curva Precision-Recall
    PrecisionRecallDisplay.from_estimator(modelo, X_test, y_test, ax=axes[2])
    axes[2].set_title('Curva Precision-Recall')
    
    plt.tight_layout()
    plt.show()

# 3. Construcción de la Tabla Comparativa Final (Estructura solicitada / FOTO)
df_tabla_final = pd.DataFrame(resultados_prueba).T
df_tabla_final = df_tabla_final[['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']]
df_tabla_final.index.name = 'Modelo finalista'

print("\n" + "="*65)
print("TABLA COMPARATIVA FINAL (CONJUNTO DE PRUEBA)")
print("="*65)
display(df_tabla_final.round(4))

**Análisis de la Evaluación en Prueba**

***Confirmación de resultados y Sobreajuste:***

El desempeño de ambos modelos se mantuvo sumamente consistente. El Modelo A (Random Forest) alcanzó un ROC-AUC de 0.8433 en prueba, mientras que el Modelo B (Regresión Logística) logró un 0.8457. Estos valores son prácticamente idénticos a los obtenidos durante la fase de validación cruzada, lo que demuestra una pérdida mínima y confirma que ninguno de los algoritmos sufre de sobreajuste (overfitting).

***Desempeño en las clases:***

Al analizar las matrices de confusión utilizando el umbral predeterminado (0.50), notamos una diferencia de comportamiento. El Modelo A priorizó la detección, logrando identificar a 297 clientes en riesgo (Recall de 0.7941) a costa de generar más falsas alarmas (274 Falsos Positivos). Por el contrario, el Modelo B (Regresión Logística) fue más conservador en este umbral inicial: aunque dejó escapar a 175 clientes reales en riesgo (desplomando su Recall a 0.5321), logró una mejor Precisión (0.6633) y Exactitud (0.8041), cometiendo menos de la mitad de falsas alarmas (solo 101 Falsos Positivos).

***Selección definitiva:***

Aunque en un escenario de retención (Churn) es financieramente prioritario detectar a tiempo a quienes se van a ir (lo que superficialmente daría la victoria al Modelo A), **se selecciona el Modelo B (Regresión Logística)** como el modelo definitivo. La razón analítica es que las probabilidades del Modelo B están excelentemente calibradas, lo que nos permite ajustar el umbral operativo (como se abordará en el Paso 17) para elevar su Recall a los niveles necesarios para el negocio. Al hacer esto, obtenemos lo mejor de ambos mundos: la capacidad de detección requerida, sumada a una interpretabilidad total de los coeficientes, menor costo computacional y una facilidad de operación en producción que el Random Forest (al ser un modelo de "caja negra") no puede ofrecer.

# Paso 16: Analizar los errores finales

**Tabla de Ejemplos Representativos de Errores**

| Caso | Clase real | Predicción | Probabilidad (Churn) | Tipo de error | Características relevantes |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Caso 1** | 0 (No Churn) | 1 (Churn) | 0.58 | **Falso Positivo (FP)** | Contrato *Month-to-month*, pago por *Electronic check*, cargos mensuales altos ($85.5 USD), y *tenure* corto (3 meses). El modelo asume alto riesgo por el perfil, pero el cliente decidió quedarse. |
| **Caso 2** | 1 (Churn) | 0 (No Churn) | 0.42 | **Falso Negativo (FN)** | Contrato *Two year*, *tenure* largo (58 meses), *Total_Servicios* alto. El modelo confía en su alta lealtad histórica, pero el cliente canceló de forma imprevista. |

**Análisis Cualitativo de los Errores**

#### 1. Patrones compartidos en los casos mal clasificados
* **Falsos Positivos (FP):** El modelo tiende a equivocarse penalizando en exceso a los clientes que tienen contratos mes a mes (`Month-to-month`) y métodos de pago electrónicos (`Electronic check`). Aunque estadísticamente este grupo abandona más, hay una fracción que muestra resistencia al cambio y permanece activa, lo que genera falsas alertas.
* **Falsos Negativos (FN):** Ocurren predominantemente en perfiles que el algoritmo cataloga como "seguros" debido a su alta antigüedad (`tenure`) o contratos de larga duración (`One year` o `Two year`). El error surge porque estos clientes abandonan por factores externos imprevisibles (como mudanza de zona, cambio drástico de tarifas o insatisfacción puntual con el soporte técnico) que no se reflejan en las variables demográficas o de facturación básica.

#### 2. Variables poco representadas o sesgos del modelo
* Las categorías minoritarias en métodos de pago menos comunes o combinaciones atípicas (por ejemplo, clientes con antigüedad muy alta pero con cargos mensuales muy bajos) tienen menor representación, lo que provoca que el árbol de decisión tome reglas generalistas que fallan en los extremos.

#### 3. Perfiles específicos donde falla el modelo
* **Clientes "Nuevos de alto riesgo aparente":** Aquellos que recién contrataron el servicio pero que, a diferencia de la media que cancela, están dispuestos a mantenerlo a pesar de pagar tarifas elevadas.
* **Clientes "Veteranos inestables":** Usuarios con años de antigüedad que experimentan una ruptura abrupta en su comportamiento de consumo o que sufren un evento de fricción no capturado, sorprendiendo al modelo que confiaba ciegamente en su historial de retención.

#### 4. Acciones recomendadas para reducir estos errores
* **Incorporar variables de interacción o de servicio al cliente:** Integrar métricas sobre el número de reclamaciones o llamadas al soporte técnico (*tech support tickets*), ya que un cliente veterano suele manifestar su descontento por esta vía antes de cancelar.
* **Optimización de umbrales de decisión:** En lugar de usar el umbral predeterminado de 0.50, calibrar un umbral de probabilidad personalizado mediante la curva Precision-Recall para equilibrar el costo de perder un cliente frente al costo operativo de lanzar campañas de retención a Falsos Positivos.
* **Recopilación de datos cualitativos:** Añadir encuestas de salida para entender los motivos de cancelación no monetarios (mudanzas, ofertas de la competencia), reduciendo los Falsos Negativos originados por causas externas al comportamiento digital o de pago.

# Paso 17: Optimizar el umbral de decisión

In [ ]:
%whos

In [ ]:
print(nombre)
print(modelo_b)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (precision_recall_curve, roc_curve,
                              precision_score, recall_score, f1_score,
                              confusion_matrix)
import matplotlib.pyplot as plt

# --- Probabilidades predichas con el modelo ya entrenado (Modelo B) ---
y_proba_b = modelo_b.predict_proba(X_test)[:, 1]

# ============================================================
# 1. Evaluación de varios umbrales con distintos criterios
# ============================================================
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_proba_b)
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba_b)

# --- Índice de Youden ---
youden_index = tpr - fpr
umbral_youden = thresholds_roc[np.argmax(youden_index)]
print(f"Umbral óptimo según Índice de Youden: {umbral_youden:.3f}")

# --- Umbral que maximiza F1-score ---
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
umbral_f1_max = thresholds_pr[np.argmax(f1_scores[:-1])]
print(f"Umbral óptimo según F1-score máximo: {umbral_f1_max:.3f}")

# --- Umbral con restricción mínima de recall (ej. Recall >= 0.70) ---
recall_minimo = 0.70
idx_validos = np.where(recalls[:-1] >= recall_minimo)[0]
if len(idx_validos) > 0:
    umbral_recall_min = thresholds_pr[idx_validos[-1]]
    print(f"Umbral con Recall >= {recall_minimo}: {umbral_recall_min:.3f}")

# ============================================================
# 2. Tabla comparativa de umbrales
# ============================================================
umbrales_a_evaluar = sorted(set([0.30, 0.40, 0.50, round(umbral_youden,2), 0.60, 0.70]))

resultados = []
for u in umbrales_a_evaluar:
    y_pred_u = (y_proba_b >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    resultados.append({
        'Umbral': u,
        'Precision': round(precision_score(y_test, y_pred_u), 4),
        'Recall': round(recall_score(y_test, y_pred_u), 4),
        'F1-score': round(f1_score(y_test, y_pred_u), 4),
        'Falsos_positivos': fp,
        'Falsos_negativos': fn
    })

tabla_umbrales = pd.DataFrame(resultados)
print(tabla_umbrales)

# --- Gráfica Precision-Recall vs Umbral ---
plt.figure(figsize=(8,5))
plt.plot(thresholds_pr, precisions[:-1], label='Precision')
plt.plot(thresholds_pr, recalls[:-1], label='Recall')
plt.axvline(x=umbral_youden, color='gray', linestyle='--', label=f'Youden ({umbral_youden:.2f})')
plt.xlabel('Umbral')
plt.ylabel('Score')
plt.title('Precision y Recall vs. Umbral de decisión — Modelo B (Regresión Logística)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

| Umbral | Precision | Recall | F1-score | Falsos positivos | Falsos negativos |
|--------|-----------|--------|----------|-------------------|-------------------|
| 0.26   | 0.5067    | 0.8048 | 0.6219   | 293               | 73                |
| 0.30   | 0.5337    | 0.7620 | 0.6278   | 249               | 89                |
| 0.40   | 0.5829    | 0.6578 | 0.6181   | 176               | 128               |
| 0.50   | 0.6633    | 0.5321 | 0.5905   | 101               | 175               |
| 0.60   | 0.7024    | 0.3850 | 0.4974   | 61                | 230               |
| 0.70   | 0.8034    | 0.2513 | 0.3829   | 23                | 280               |

Umbral operativo seleccionado 0.30

Objetivo que prioriza: Este umbral prioriza el Recall (0.7620) sobre la precision (0.5337), es decir, busca detectar la mayor cantidad posible de clientes que realmente van a abandonar el servicio, aun a costa de generar más flasas alarmas

Costo que reduce: Reduce el costo asociado a los falsos negativos (clientes que se van sin que el modelo los detecte). En un problema de churn, no detectar a un cliente en riesgo significa perder la oportunidad de retenerlo; un costo típicamente mayor que el de una campaña de retención dirigida a alguien que en realidad no iba a irse

Consecuencias que pueden generar: Con este umbral se generan 249 falsos positivos (clientes marcados como riesgo que en realidad no iban a cancelar). Esto implica un gasto adicional en acciones de retención (descuentos, llamadas, promociones) dirigidas a clientes que no las necesitaban, lo cual reduce la eficiencia de las campañas si no se controla el volumen

Por qué no debe adoptarse sin revisión organizacional: La elección de este umbral asume que el costo de perder un cliente es mayor que el costo de una acción de retención innecesaria, pero esta relación de costos depende del negocio real (margen por cliente, costo de las campañas de retención, capacidad operativa del equipo de retencio=ón, etc), información que no está disponible en este análisis. Por eso, antes de operacionalizar este umbral, el equipo de negocio debe validar si esta relación de costos es correcta para el contexto específico de la empresa y ajustar el umbral si es necesario

# Paso 18: Seleccionar el modelo final

### Regresión Logística

| Criterio | Evidencia | Conclusión |
| :--- | :--- | :--- |
| **Desempeño** | La Regresión Logística alcanzó el ROC-AUC más alto en la evaluación cruzada con un valor de 0.8489. Esto supera a modelos más complejos como el Random Forest (0.8264) y el SVM con kernel RBF (0.8289). | Es el algoritmo con la mejor capacidad matemática probada para distinguir exitosamente entre las clases de clientes que abandonan y los que se quedan. |
| **Generalización** | Durante la validación cruzada RepeatedStratifiedKFold (5x3), la Regresión Logística mostró una desviación estándar de apenas 0.0116. | El modelo es altamente estable y tiene una gran capacidad de generalizar frente a datos no vistos, garantizando consistencia en producción. |
| **Errores** | Presenta un Recall de 0.5382 y un F1-Score de 0.5956. Aunque su Recall es menor al del SVM Lineal (0.8201), mantiene una mejor Precision general de 0.6687. | Ofrece un balance adecuado; aunque detecta menos abandonos totales que el SVM Lineal, comete menos errores de falsos positivos, lo que optimiza el presupuesto en campañas de retención. |
| **Interpretabilidad** | Es un modelo lineal que ofrece interpretabilidad a través de coeficientes exactos, a diferencia del Random Forest que opera como una "caja negra". | Facilita la justificación y auditoría de cada predicción individual ante los tomadores de decisiones del negocio, explicando qué factores influyen en el riesgo. |
| **Complejidad** | El tiempo de entrenamiento fue de solo 0.1935 segundos, considerablemente inferior a los 1.1337 segundos del Random Forest y los 2.3463 del SVM RBF. | Es computacionalmente económico, lo que agiliza el reentrenamiento futuro del modelo a medida que se inyectan nuevos datos en el pipeline. |
| **Operación** | El tiempo de inferencia es extremadamente rápido, promediando 0.0491 segundos frente a los 0.5733 segundos requeridos por el SVM RBF. | Su baja latencia lo hace ideal para implementaciones en tiempo real o arquitecturas ETL de alta demanda. |
| **Riesgo** | Tiene un menor riesgo de sobreajuste (overfitting) en comparación con un árbol de decisión sin restricciones o un modelo con fronteras de decisión complejas. | Representa una apuesta analítica segura y controlada que no memoriza el ruido de los datos de entrenamiento. |

La Regresión Logística permite extraer y auditar los coeficientes matemáticos exactos de cada variable. Esto significa que podemos explicar con precisión cómo factores críticos, como el tipo de contrato, el método de pago o la antigüedad del cliente (tenure), incrementan o reducen la probabilidad de abandono.

Adémas, al conocer la ecuación exacta de la regresión, es posible construir tableros interactivos que vayan más allá del análisis histórico. Se pueden implementar expresiones DAX avanzadas y parámetros What-If para ver en tiempo real cómo cambiaría la probabilidad de retención de un segmento de clientes si se modifica su oferta actual.

Por ultimo, con un tiempo de entrenamiento de apenas 0.19 segundos y una inferencia casi instantánea de 0.049 segundos, la Regresión Logística garantiza que el procesamiento de nuevos lotes masivos de datos requiera recursos de cómputo mínimos, reduciendo costos operativos y asegurando que los pipelines de datos se ejecuten sin retrasos.

# Paso 19: Interpretar el modelo globalmente

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Extraer nombres de las variables tras el preprocesamiento ---
preprocesador_b = modelo_b.named_steps['preprocesador']
nombres_features = preprocesador_b.get_feature_names_out()

# --- Extraer coeficientes del modelo entrenado ---
coeficientes = modelo_b.named_steps['modelo'].coef_[0]

# --- Armar tabla de ranking ---
df_coef = pd.DataFrame({
    'Variable': nombres_features,
    'Coeficiente': coeficientes,
    'Coeficiente_absoluto': np.abs(coeficientes)
}).sort_values('Coeficiente_absoluto', ascending=False).reset_index(drop=True)

print("Ranking de variables por importancia (valor absoluto del coeficiente):")
print(df_coef.head(15))

# --- Visualización: top 15 variables más influyentes ---
top_n = 15
df_plot = df_coef.head(top_n).sort_values('Coeficiente')

plt.figure(figsize=(9,7))
colores = ['#d62728' if c < 0 else '#2ca02c' for c in df_plot['Coeficiente']]
plt.barh(df_plot['Variable'], df_plot['Coeficiente'], color=colores)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coeficiente (Regresión Logística)')
plt.title(f'Top {top_n} variables más influyentes en la predicción de Churn')
plt.tight_layout()
plt.show()

Interpretación de los principales factores:

Tipo de contrato(Contract): es el factor más influyente. Como está codificado ordinalmente (Month-to-Month=0, One year=1, Two year=2), un coeficiente negativo significa que entre más largo el contrato, menor la probabilidad de churn, lo cual tiene sentido: clientes con compromisos más largos tienden a quedarse

Cliente nuevo(Cliente_Nuevo): es la segunda variable más influyente, y aumenta el riesgo de churn. Esto es coherente con el patrón típico de negocio: los clientes recién adquiridos son más propensos a cancelar, ya que aun no están afianzados con el servicio

Tipo de internet: tener Fibra óptica aumenta el riesgo de churn, mientras que tener DLS lo reduce(comparado con no tener internet, la categoría base). Esto podría deberse a que el servicio de fibra suele ser más caro o tener más problemas percibidos de calidad/precio

Antigüedad(tenure): entre más tiempo lleva el cliente, menor la probabilidad de abandono; un patrón muy esperado y consistente con la lógica de negocio

Cargo mensual(MonthlyCharges): el signo negativo aquí es un poco contraintuitivo(esperaríamos que pagar más se asocie con más churn), y probablemente se debe a colinealidad con otras variables como "InternetService" o "Contract"; el modelo "reparte el efecto entre variables correlacionadas

Detalle notable: todas las variables "_No internet service"(TechSupport, StreamingMovies, StreamingTV, DeviceProtection, OnlineBackup, OnlineSecurity) tienen exactamente el mismo coeficiente (-0.3035). Esto no es casualidad; es un artefacto de colinalidad perfecta: todas esas categorías coinciden exactamente con no tener "InternetService", así que el modelo no puede distinguir su efecto individual

Limitaciones del método:

Los coeficientes de una Regresión Logística asumen una relación log-lineal entre las variables (transformadas) y la probabilidad de churn; no capturan interacciiones complejas entre variables ni relaciones no lineales

Son sensibles a la escala de las variables (por eso se estandarizaron las numéricas) y a la colinalidad (como el caso de "No internet service" arriba), lo cual puede distorsionar la magnitud e incluso el signo de un coeficiente

El coeficiente refleja el efecto de la variables controlando por las demás, no su efecto aislado; por eso "MonthlyChrages" puede verse contraintuitivo.

La influencia de una variables en el modelo no implica causalidad; el modelo solo captura asociaciones estadísticas en los datos históricos, no relaciones causa-efecto. Por ejemplo, que "Fibra óptica" se asocie a más churn no significa que la fibra cause el abandono; podría deberse a ffactores de precio, competencia u otras variables no incluidas

# Paso 20: Explicar predicciones individuales

In [ ]:
%pip install shap

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# pip install shap 
# En caso de no tenerlo instalado descomentar la línea anterior y ejecutar gracias

# 1. Obtener predicciones con el umbral operativo optimizado (0.30) del Paso 17
umbral_operativo = 0.30
y_proba_final = modelo_b.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba_final >= umbral_operativo).astype(int)

# 2. Identificar los índices de los 4 casos solicitados (TP, TN, FP, FN)
y_test_arr = np.array(y_test)
idx_tp = np.where((y_test_arr == 1) & (y_pred_final == 1))[0][0] # Verdadero Positivo
idx_tn = np.where((y_test_arr == 0) & (y_pred_final == 0))[0][0] # Verdadero Negativo
idx_fp = np.where((y_test_arr == 0) & (y_pred_final == 1))[0][0] # Falso Positivo
idx_fn = np.where((y_test_arr == 1) & (y_pred_final == 0))[0][0] # Falso Negativo

casos_indices = {
    'Verdadero Positivo (Fuga correctamente detectada)': idx_tp,
    'Verdadero Negativo (Cliente leal correctamente detectado)': idx_tn,
    'Falso Positivo (Falsa alarma de fuga)': idx_fp,
    'Falso Negativo (Fuga no detectada)': idx_fn
}

# 3. Preparar los datos transformados para SHAP
# Extraemos el preprocesador y el modelo final (Regresión Logística)
preprocesador = modelo_b.named_steps['preprocesador']
modelo_logreg = modelo_b.named_steps['modelo']
nombres_variables = preprocesador.get_feature_names_out()

# Transformamos X_test para que SHAP entienda los números escalados
X_test_prep = preprocesador.transform(X_test)
X_test_prep_df = pd.DataFrame(X_test_prep, columns=nombres_variables)

# 4. Configurar SHAP LinearExplainer
print("Calculando valores SHAP para la Regresión Logística...")
explainer = shap.LinearExplainer(modelo_logreg, X_test_prep_df)
shap_values = explainer(X_test_prep_df)

# 5. Generar las explicaciones locales (Waterfall plots) para los 4 casos
for nombre_caso, idx in casos_indices.items():
    print(f"\n{'='*70}")
    print(f"CASO: {nombre_caso}")
    print(f"Probabilidad predicha de Churn: {y_proba_final[idx]:.4f} (Umbral de corte: {umbral_operativo})")
    print(f"Resultado Real: {y_test_arr[idx]} | Predicción del Modelo: {y_pred_final[idx]}")
    print(f"{'='*70}")
    
    # Mostrar la gráfica tipo Cascada (Waterfall)
    plt.figure()
    shap.plots.waterfall(shap_values[idx], max_display=10, show=True)

### Tabla de Predicciones Individuales

| Caso | Resultado real | Predicción | Factores principales (Variables SHAP) | Interpretación (Análisis local) |
| :--- | :--- | :--- | :--- | :--- |
| **Verdadero positivo** | 1 (Canceló) | 1 (Riesgo alto: 0.3175) | **Riesgo (+):** `ord__Contract` = 0 (+0.51), `num__tenure` bajo (+0.37).<br>**Retención (-):** `nom__InternetService_DSL` = 1 (-0.48). | El modelo superó el umbral de 0.30 y detectó la fuga exitosamente porque el cliente tenía un contrato mes a mes (`ord__Contract=0`) y baja antigüedad. Aunque tener internet DSL jalaba la predicción hacia la lealtad, la inestabilidad del contrato corto tuvo mucho más peso. |
| **Verdadero negativo** | 0 (Se quedó) | 0 (Riesgo bajo: 0.0443) | **Retención (-):** `num__tenure` alto (-1.11), `ord__Contract` = 2 (-0.98), `num__MonthlyCharges` alto (-0.95).<br>**Riesgo (+):** `nom__InternetService_Fiber optic` = 1 (+0.37). | El modelo clasificó al cliente como altamente seguro (solo 4% de riesgo) gracias a su fuerte anclaje: tiene alta antigüedad y un contrato a dos años (`ord__Contract=2`). Estos factores de lealtad aplastaron por completo el ligero riesgo que le generaba tener el servicio de Fibra Óptica. |
| **Falso positivo** | 0 (Se quedó) | 1 (Riesgo alto: 0.6383) | **Riesgo (+):** `num__tenure` muy bajo (+0.62), `ord__Contract` = 0 (+0.51), `nom__InternetService_Fiber optic` = 1 (+0.37). | El modelo generó una fuerte falsa alarma clasificándolo con un 63% de riesgo. Se dejó llevar por el "combo de riesgo" clásico: muy baja antigüedad, contrato mensual y servicio de fibra óptica. El cliente decidió quedarse por factores externos a sus datos demográficos. |
| **Falso negativo** | 1 (Canceló) | 0 (Riesgo bajo: 0.1719) | **Retención (-):** `nom__InternetService_DSL` = 1 (-0.48), `nom__TechSupport_Yes` = 1 (-0.18), `bin__Cliente_Nuevo` = 0 (-0.14).<br>**Riesgo (+):** `ord__Contract` = 0 (+0.51). | El modelo no vio venir esta cancelación. Fue "engañado" por características que normalmente indican lealtad: el cliente usaba DSL, tenía soporte técnico activo y no era un cliente nuevo. El único foco rojo era su contrato mes a mes, pero no alcanzó a sumar la fuerza necesaria para superar el umbral de 0.30. |

# Paso 21: Evaluar posibles sesgos

Para evaluar posibles sesgos en el modelo, analizaremos el desempeño de la Regresión Logística (utilizando el umbral operativo optimizado de 0.30) segmentando a la población según la variable creada en la ingeniería de características: **Antigüedad del cliente (`Cliente_Nuevo`)**. 

Dividiremos el conjunto de prueba en dos grupos:
* **Clientes Nuevos:** Antigüedad $\le$ 6 meses.
* **Clientes Establecidos:** Antigüedad > 6 meses.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 1. Definir el umbral operativo elegido en el Paso 17
umbral = 0.30

# 2. Obtener probabilidades predichas del Modelo B (Regresión Logística) en X_test
y_proba = modelo_b.predict_proba(X_test)[:, 1]
y_pred_umbral = (y_proba >= umbral).astype(int)

# 3. Separar los índices de los grupos basados en la característica 'Cliente_Nuevo'
# X_test ya tiene esta columna gracias a la ingeniería de características del Paso 5
mascara_nuevos = X_test['Cliente_Nuevo'] == 1
mascara_establecidos = X_test['Cliente_Nuevo'] == 0

resultados_sesgos = []

# 4. Función para calcular métricas por segmento poblacional
def calcular_metricas_grupo(nombre_grupo, mascara):
    y_test_grupo = y_test[mascara]
    y_pred_grupo = y_pred_umbral[mascara]
    
    tn, fp, fn, tp = confusion_matrix(y_test_grupo, y_pred_grupo).ravel()
    
    tamaño = len(y_test_grupo)
    tasa_abandono = (y_test_grupo.sum() / tamaño) * 100
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    resultados_sesgos.append({
        'Grupo': nombre_grupo,
        'Tamaño': tamaño,
        'Tasa de abandono (%)': round(tasa_abandono, 2),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'FPR (Tasa Falsos Positivos)': round(fpr, 4),
        'FNR (Tasa Falsos Negativos)': round(fnr, 4)
    })

# 5. Ejecutar los cálculos para ambos grupos
calcular_metricas_grupo('Clientes Nuevos (<= 6 meses)', mascara_nuevos)
calcular_metricas_grupo('Clientes Establecidos (> 6 meses)', mascara_establecidos)

# 6. Mostrar la tabla resultante para el análisis de sesgos
df_sesgos = pd.DataFrame(resultados_sesgos)
display(df_sesgos)

### Análisis de Diferencias y Mitigación

**1. Diferencias observadas:**
Existe una diferencia estadística marcada en el comportamiento del modelo entre ambos grupos. Para los clientes nuevos, el modelo es sumamente agresivo: logra identificar casi todas las fugas reales (*Recall* del 0.8728), pero dispara falsas alarmas en más de la mitad de los que deciden quedarse (FPR de 0.5942). En contraste, con los clientes establecidos, el modelo es mucho más conservador, dejando escapar casi una tercera parte de las fugas reales (FNR de 0.3333).

**2. Factores generadores:**
Esta diferencia no obedece a un sesgo discriminatorio injusto, sino a una disparidad en las tasas base de abandono y en la naturaleza del problema. Los clientes nuevos están en un periodo de "prueba" y su probabilidad natural de cancelar es altísima (55.63%). El modelo aprende que ser nuevo es un fuerte predictor de abandono. Por el contrario, un cliente con años de antigüedad tiene una inercia fuerte hacia la retención; cuando cancelan, suele deberse a factores disruptivos (fallas de servicio, mudanzas, ofertas agresivas de la competencia) que no están representados en las variables estáticas de facturación.

**3. Limitaciones del análisis:**
El análisis está limitado por la falta de variables dinámicas de interacción (como tickets de soporte recientes, quejas o uso diario del servicio). Al no tener estos datos, el modelo asume que un cliente veterano siempre será leal basándose únicamente en su historial, lo que infla artificialmente la tasa de Falsos Negativos en ese grupo.

**4. Medidas de revisión y mitigación:**
* **No concluir discriminación:** La diferencia estadística refleja un sesgo algorítmico basado en el ciclo de vida del cliente, no una discriminación ética o demográfica.
* **Umbrales dinámicos (Mitigación):** Una acción operativa viable es aplicar umbrales de decisión distintos por segmento. Por ejemplo, mantener un umbral estricto para clientes nuevos (para contener el gasto en falsas alarmas) y relajar el umbral aún más para clientes establecidos, asegurando que se capturen las fugas de las cuentas más valiosas.
* **Enriquecimiento de datos:** Incorporar variables temporales recientes (ej. "Llamadas a soporte en los últimos 30 días") para dar al modelo señales tempranas de insatisfacción en clientes veteranos.

# Paso 22: Documentar riesgos y condiciones de uso 

### Registro de riesgos — Modelo B (Regresión Logística, umbral 0.30)

| Riesgo | Causa | Consecuencia | Probabilidad | Impacto | Mitigación |
|---|---|---|---|---|---|
| Falsos negativos | El modelo no detecta a todos los clientes que abandonan (Recall 0.7620; 89 falsos negativos en 1409 registros de prueba, cerca de 1 de cada 4 clientes que se van). | Se pierden clientes sin oportunidad de retención, con pérdida de ingresos recurrentes. | Media | Alto | Usar un umbral bajo (0.30) para priorizar Recall; complementar con reglas de negocio para clientes de alto valor; monitorear el Recall en producción. |
| Falsos positivos | Al bajar el umbral se marcan más clientes como riesgo (Precision 0.5337; 249 falsos positivos, alrededor del 17.7% de los registros de prueba). | Gasto en campañas de retención (descuentos, llamadas) para clientes que no iban a irse; posible saturación del equipo de retención. | Alta | Medio | Priorizar la atención por probabilidad predicha (no tratar a todos igual); limitar el volumen de campañas a la capacidad operativa; medir el costo por retención. |
| Drift | El dataset es una foto histórica; el comportamiento de los clientes, precios y competencia cambian con el tiempo. | El desempeño se degrada sin que nadie lo note y los umbrales dejan de ser válidos. | Media | Alto | Monitorear periódicamente Recall, Precision y la distribución de variables (tenure, MonthlyCharges, Contract); reentrenar y recalibrar el umbral cuando haya degradación. |
| Datos incompletos | `TotalCharges` tenía 11 valores nulos ocultos (clientes con tenure 0) imputados con 0; en producción pueden llegar campos vacíos o categorías nuevas. | Predicciones poco confiables para registros incompletos. | Media | Medio | Mantener la imputación dentro del pipeline; validar el esquema y los nulos antes de predecir; enviar a revisión humana los registros con datos críticos faltantes. |
| Uso automático | Ejecutar acciones (descuentos, cancelaciones, contacto) solo con la salida del modelo, sin supervisión. | Decisiones erróneas a escala, con costos y afectación a clientes, dado que casi la mitad de las alertas serían falsas. | Media | Alto | Usar el modelo solo como apoyo a la decisión, con revisión humana antes de cualquier acción; registrar las decisiones tomadas. |
| Sesgo | El modelo rinde distinto según la antigüedad del cliente. En clientes nuevos (≤ 6 meses, n=311, abandono 55.63%): Recall 0.8728, FPR 0.5942. En establecidos (> 6 meses, n=1098, abandono 18.31%): Recall 0.6667, FNR 0.3333, Precision 0.4452. La diferencia se explica por las distintas tasas base de abandono y por la influencia de `Cliente_Nuevo` en el modelo. No se evaluó el desempeño por género ni por `SeniorCitizen`, variables que el modelo sí usa. | Más falsas alarmas en clientes nuevos y más abandonos no detectados en clientes establecidos. Posible tratamiento desigual de grupos demográficos aún no medido. | Media | Alto | Aplicar umbrales por segmento de antigüedad; enriquecer los datos con variables recientes (por ejemplo, llamadas a soporte de los últimos 30 días); evaluar Recall y Precision por género y `SeniorCitizen` antes de operar; revisión organizacional. |

Usos permitidos
- Identificar clientes con mayor probabilidad de abandono para priorizar campañas de retención.
- Analizar los factores asociados al churn (coeficientes) para apoyar decisiones de negocio.
- Comparar escenarios de umbral según capacidad operativa y costo de retención.

Usos no recomendados
- Tomar decisiones automáticas sobre el cliente (cancelar servicios, negar ofertas, cambiar precios) sin revisión humana.
- Interpretar los coeficientes como relaciones causales.
- Usarlo en poblaciones o mercados distintos a los del dataset, o con datos muy posteriores a los de entrenamiento, sin revalidar.
- Evaluar o sancionar a personas o empleados con base en sus predicciones.

Necesidad de revisión humana
- Toda acción de retención basada en el modelo debe ser validada por el equipo de retención o negocio.
- Se requiere revisión adicional para clientes con datos incompletos o de alto valor.
- El umbral operativo (0.30) requiere validación organizacional de costos antes de adoptarse.

Responsables de supervisión
- Responsable del modelo (científico de datos): monitoreo de métricas y reentrenamiento.
- Responsable de negocio o retención: validación del umbral y de las campañas.
- Responsable de cumplimiento o ética de datos: revisión de sesgos y uso de variables sensibles.

Condiciones para suspender el modelo
- Caída sostenida del Recall o de la Precision respecto a los valores de referencia (Recall 0.7620 y Precision 0.5337 con umbral 0.30), con un límite definido por el equipo.
- Cambios importantes en la distribución de los datos (drift) o en el negocio (nuevos planes, precios o competencia).
- Detección de disparidades relevantes entre subgrupos (antigüedad, género, adultos mayores).
- Aumento de datos incompletos o inconsistentes en la entrada.
- Cambios en la regulación o política de uso de datos personales.

# Paso 23: Serializar el pipeline y el modelo

In [ ]:
import joblib
import json

# 1. Serializar el Pipeline Final Completo (Preprocesamiento + Modelo)
joblib.dump(modelo_b, 'pipeline_final_churn.joblib')

# 2. Serializar el Modelo de forma aislada
modelo_logreg = modelo_b.named_steps['modelo']
joblib.dump(modelo_logreg, 'modelo_logreg_churn.joblib')

# 3. Serializar la configuración del Umbral Óptimo (0.30 del Paso 17)
config_umbral = {'umbral_operativo': 0.30, 'metrica_optimizada': 'F1-score / Recall'}
with open('umbral_churn.json', 'w') as f:
   json.dump(config_umbral, f)

print("✅ Artefactos serializados correctamente en el directorio de trabajo.")

## Paso 23: Serialización del modelo

A continuación se documentan los artefactos generados para el despliegue del modelo de predicción de abandono (Churn).

| Artefacto | Archivo | Versión | Fecha | Biblioteca |
| :--- | :--- | :--- | :--- | :--- |
| **Pipeline final** | `pipeline_final_churn.joblib` | v1.0 | 2026-09-27 | `joblib` (scikit-learn) |
| **Modelo** | `modelo_logreg_churn.joblib` | v1.0 | 2026-09-27 | `joblib` (scikit-learn) |
| **Codificación de objetivo** | *N/A (Nativa)* | N/A | 2026-09-27 | *Variable binaria directa (0/1)* |
| **Umbral** | `umbral_churn.json` | v1.0 | 2026-09-27 | `json` (Python Standard Library) |

> ⚠️ **ADVERTENCIA DE SEGURIDAD:**
> Los archivos serializados generados con `joblib` o `pickle` pueden contener código ejecutable malicioso. Al usar funciones como `joblib.load()`, el entorno de Python ejecuta las instrucciones de reconstrucción del objeto sin ninguna restricción. **Nunca** cargues ni deserialices archivos `.joblib`, `.pkl` o similares que provengan de fuentes desconocidas, correos electrónicos no verificados o repositorios de terceros sin auditar, ya que esto podría comprometer por completo la seguridad del sistema anfitrión.

## Paso 24: Script de Inferencia y Validación Independiente para Predicción de Churn



In [ ]:
import json
import joblib
import pandas as pd
import numpy as np

def cargar_artefactos():
    """Carga el pipeline final serializado y la configuración del umbral."""
    pipeline = joblib.load('pipeline_final_churn.joblib')
    with open('umbral_churn.json', 'r') as f:
        config_umbral = json.load(f)
    return pipeline, config_umbral['umbral_operativo']

def aplicar_ingenieria_caracteristicas(df):
    """Aplica la misma ingeniería de características definida en el Paso 5."""
    df_temp = df.copy()
    
    # Total de servicios contratados
    columnas_servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                          'TechSupport', 'StreamingTV', 'StreamingMovies']
    df_temp['Total_Servicios'] = (df_temp[columnas_servicios] == 'Yes').sum(axis=1)
    
    # Indicador de cliente nuevo (6 meses o menos)
    df_temp['Cliente_Nuevo'] = (df_temp['tenure'] <= 6).astype(int)
    
    return df_temp

def validar_estructura(df):
    """Valida que la estructura de entrada contenga todas las columnas obligatorias."""
    columnas_requeridas = [
        'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
        'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
        'MonthlyCharges', 'TotalCharges'
    ]
    
    faltantes = [col for col in columnas_requeridas if col not in df.columns]
    if faltantes:
        raise ValueError(f"Error de estructura: Faltan las columnas requeridas: {faltantes}")
    return True

def predecir_nuevos_registros(df_nuevos):
    """Ejecuta el flujo completo de validación, preprocesamiento, inferencia y umbral."""
    pipeline, umbral = cargar_artefactos()
    validar_estructura(df_nuevos)
    
    df_trabajo = df_nuevos.copy()
    df_trabajo['TotalCharges'] = pd.to_numeric(df_trabajo['TotalCharges'], errors='coerce').fillna(0)
    
    df_procesado = aplicar_ingenieria_caracteristicas(df_trabajo)
    y_proba = pipeline.predict_proba(df_procesado)[:, 1]
    y_pred = (y_proba >= umbral).astype(int)
    
    resultados = pd.DataFrame({
        'Probabilidad_Churn': y_proba,
        'Clase_Predicha': y_pred,
        'Decision': np.where(y_pred == 1, 'Alerta de Abandono (Churn)', 'Cliente Retenido')
    })
    
    return resultados

if __name__ == "__main__":
    nuevo_cliente = pd.DataFrame([{
        'gender': 'Female',
        'SeniorCitizen': 0,
        'Partner': 'Yes',
        'Dependents': 'No',
        'tenure': 2,
        'PhoneService': 'Yes',
        'MultipleLines': 'No',
        'InternetService': 'Fiber optic',
        'OnlineSecurity': 'No',
        'OnlineBackup': 'No',
        'DeviceProtection': 'No',
        'TechSupport': 'No',
        'StreamingTV': 'No',
        'StreamingMovies': 'No',
        'Contract': 'Month-to-month',
        'PaperlessBilling': 'Yes',
        'PaymentMethod': 'Electronic check',
        'MonthlyCharges': 70.70,
        'TotalCharges': 151.50
    }])
    
    res = predecir_nuevos_registros(nuevo_cliente)
    print("Resultado de la predicción:")
    print(res)

# Paso 25: Implementar un mecanismo mínimo de inferencia

Se creo un script ejecutable desde la consola o terminal usando el siguiente comando de Powershell:

**python Proyecto_Final/predictor.py**

La información que recibe del cliente es mediante un formato estructurado (JSON).

El Sript Carga el modelo (pipeline_final_churn.joblib), aplica las transformaciones necesarias a los datos estructurados recibidos y entrega la predicción correspondiente.

Ejemplo de salida:

**--- RESULTADO DE LA INFERENCIA ---**

**Probabilidad estimada de abandono: 48.69%**

**Clasificación resultante: Abandono (Churn)**

**Umbral utilizado: 0.3**

**Fecha / Versión del modelo: v1.0 - Septiembre 2026**

**Aviso: El resultado mostrado es una estimación basada en un modelo estadístico y requiere revisión humana para la toma de decisiones.**

**----------------------------------**

# Paso 26: Implementación de Controles de Validación y Calidad de Datos

In [ ]:


def validar_datos_entrada(df):
    """
    Valida los datos de entrada recopilando anomalías sin detener la ejecución.
    Detecta: campos faltantes, tipos incorrectos, valores fuera de rango, 
    categorías no reconocidas, columnas adicionales y estructura incompleta.
    """
    reporte = {
        "valido": True,
        "campos_faltantes": [],
        "columnas_adicionales": [],
        "tipos_incorrectos": [],
        "valores_fuera_de_rango": [],
        "categorias_no_reconocidas": []
    }
    
    # 1. Estructura incompleta y columnas adicionales
    columnas_requeridas = [
        'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
        'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
        'MonthlyCharges', 'TotalCharges'
    ]
    
    cols_actuales = set(df.columns)
    cols_req = set(columnas_requeridas)
    
    faltantes = list(cols_req - cols_actuales)
    adicionales = list(cols_actuales - cols_req)
    
    if faltantes:
        reporte["campos_faltantes"] = faltantes
        reporte["valido"] = False
    if adicionales:
        reporte["columnas_adicionales"] = adicionales
        
    # Validar solo sobre columnas existentes para evitar errores fatales
    cols_a_validar = list(cols_req & cols_actuales)
    df_val = df[cols_a_validar].copy()
    
    # 2. Tipos incorrectos y conversión segura
    for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
        if col in df_val.columns:
            no_num = pd.to_numeric(df_val[col], errors='coerce').isna().sum()
            if no_num > 0:
                reporte["tipos_incorrectos"].append(f"Columna '{col}' contiene {no_num} valores no numéricos.")
                
    # 3. Valores fuera de rango (ej. tenure y cargos)
    if 'tenure' in df_val.columns:
        tenure_num = pd.to_numeric(df_val['tenure'], errors='coerce')
        fuera_rango_tenure = ((tenure_num < 0) | (tenure_num > 72)).sum()
        if fuera_rango_tenure > 0:
            reporte["valores_fuera_de_rango"].append(f"'tenure' fuera del rango [0, 72]: {fuera_rango_tenure} registros.")
            
    if 'MonthlyCharges' in df_val.columns:
        mc_num = pd.to_numeric(df_val['MonthlyCharges'], errors='coerce')
        fuera_rango_mc = (mc_num <= 0).sum()
        if fuera_rango_mc > 0:
            reporte["valores_fuera_de_rango"].append(f"'MonthlyCharges' con valores <= 0: {fuera_rango_mc} registros.")

    # 4. Categorías no reconocidas
    categorias_validas = {
        'Contract': ['Month-to-month', 'One year', 'Two year'],
        'InternetService': ['DSL', 'Fiber optic', 'No'],
        'PaymentMethod': ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
    }
    
    for col, validas in categorias_validas.items():
        if col in df_val.columns:
            invalidas = df_val[~df_val[col].isin(validas + [np.nan, None])][col].unique()
            if len(invalidas) > 0:
                reporte["categorias_no_reconocidas"].append(f"'{col}' tiene categorías no válidas: {list(invalidas)}")

    if any([reporte["campos_faltantes"], reporte["tipos_incorrectos"], reporte["valores_fuera_de_rango"], reporte["categorias_no_reconocidas"]]):
        reporte["valido"] = False

    return reporte
reporte = validar_datos_entrada(df)
print(reporte)

# Paso 27: Evaluación de Perfiles de Riesgo y Proceso de Revisión Humana

| Cliente | Probabilidad | Clasificación | Factores relevantes | Observación |
| :--- | :--- | :--- | :--- | :--- |
| **Cliente con bajo riesgo** | `0.12 (12%)` | No Churn (Bajo) | • `ord__Contract` = 2 (Contrato de dos años)<br>• `num__tenure` alto (65 meses)<br>• Pago automático por tarjeta | Cliente altamente fidelizado; no requiere intervención comercial proactiva para evitar la fuga. |
| **Cliente con riesgo medio** | `0.45 (45%)` | No Churn / En observación | • `ord__Contract` = 1 (Contrato de un año)<br>• `num__tenure` medio (14 meses)<br>• Cargos mensuales moderados | Presenta estabilidad relativa, pero muestra señales de posible cambio al aproximarse la fecha de renovación. |
| **Cliente con alto riesgo** | `0.88 (88%)` | Churn (Alto) | • `ord__Contract` = 0 (Contrato mes a mes)<br>• `nom__InternetService_Fiber optic` = 1<br>• Cargos altos y pago por cheque electrónico | Candidato prioritario para una campaña de retención urgente y asignación de un ejecutivo de cuenta. |
| **Caso con categoría poco frecuente** | `0.62 (62%)` | Churn (Moderado-Alto) | • Método de pago poco común (`Mailed check`)<br>• Ausencia de servicios de soporte técnico o seguridad | Requiere supervisión analítica para verificar si la categoría atípica está inflando artificialmente el riesgo. |
| **Caso cercano al umbral** | `0.51 (51%)` | Churn (Frontera / Indeciso) | • Probabilidad colindante al punto de corte estándar (0.50)<br>• Balance mixto entre permanencia y tipo de contrato | Caso crítico donde una pequeña variación en las variables puede inclinar la balanza; requiere doble validación humana. |

Proceso de Revisión Humana y Decisiones No Automatizadas
1. Uso operativo de cada resultado en la revisión humana:
Cliente con bajo riesgo (12%): Se excluye de las campañas masivas de retención, lo que permite optimizar el presupuesto comercial y enfocar los recursos en perfiles que realmente lo necesiten.

Cliente con riesgo medio (45%) y Caso cercano al umbral (51%): Se derivan a un equipo de análisis intermedio o supervisores de atención para revisar el historial de interacciones recientes (como llamadas a soporte técnico) antes de decidir si se les otorga una oferta preventiva.

Cliente con alto riesgo (88%) y Caso con categoría poco frecuente (62%): Se canalizan directamente con un especialista en experiencia de cliente (CX) o retención para diseñar una contraoferta personalizada, o bien, para auditar si el comportamiento responde a una anomalía en los datos.

2. Decisiones que NO deben automatizarse solo con la predicción:
Modificación o cancelación unilateral de servicios: Un sistema automatizado jamás debe restringir, alterar o dar por terminados los servicios de un cliente basándose únicamente en la probabilidad calculada por el modelo, ya que un falso positivo dañaría la relación comercial.

Otorgamiento masivo e indiscriminado de descuentos: Autorizar bonificaciones o tarifas reducidas de forma automática a todos los usuarios clasificados como "alto riesgo" puede destruir el margen financiero de la empresa en clientes que quizás no pensaban irse.

Acciones de cobranza o restricciones punitivas: Las medidas que afecten directamente la calificación o los beneficios contractuales del cliente exigen criterio humano y una verificación integral de su comportamiento financiero real para evitar sesgos algorítmicos.

# Paso 28:  Diseñar el plan de monitoreo

Para garantizar el correcto funcionamiento y la vigencia del modelo de predicción de abandono (Churn) en el entorno de producción, se establece el siguiente esquema de monitoreo continuo:

| Indicador | Frecuencia | Umbral de alerta | Acción |
| :--- | :--- | :--- | :--- |
| **Valores faltantes** | Diaria (en cada lote de inferencia) | > 5% de nulos en variables clave (ej. `tenure`, `TotalCharges`) | Revisar pipeline de datos de origen; usar imputación por defecto del pipeline y notificar a ingeniería de datos. |
| **Recall** | Mensual (requiere cruce con datos reales de abandono) | < 0.70 (Considerando el diseño óptimo de 0.76 con umbral de 0.30) | Detener campañas preventivas automáticas; recalibrar el umbral operativo o reentrenar la Regresión Logística. |
| **Data drift** | Semanal | Cambio > 10% en la distribución de variables principales (ej. `Contract`, `MonthlyCharges`) | Analizar si obedece a un cambio real en el negocio (ej. nueva oferta comercial); programar reentrenamiento si el *drift* es sostenido. |
| **Tiempo de respuesta** | Continuo (monitoreo de servidor/API) | > 2 segundos por predicción | Escalar recursos del servidor o revisar latencia en las consultas a la base de datos durante la inferencia. |
| **Categorías nuevas** | Diaria (en cada lote de inferencia) | Cualquier categoría no vista en el entrenamiento (ej. un nuevo tipo de método de pago) | Marcar registro para revisión humana; actualizar diccionarios de validación y considerar reentrenamiento para incluir la nueva categoría. |

# Paso 29: Definir criterios de actualización

| **Señal de alerta** | **Detonante / Métrica** | **Tipo de Acción** | **Responsable** | **Acción Operativa Correspondiente** | 
| :--- | :--- | :--- | :--- | :--- |
| **Caída sostenida de Recall** | Caída relativa > 10% en el Recall durante 3 evaluaciones periódicas consecutivas | **Reentrenar** | Data Scientist / MLOps Engineer | Extraer la ventana de datos más reciente, reevaluar el *Concept Drift* y reentrenar el modelo con la nueva distribución. | 
| **Aumento de Falsos Negativos** | Incremento no planificado en la fuga de clientes no identificados previamente | **Cambiar de umbral** | Data Scientist / Analista de Retención | Bajar el umbral de decisión (p. ej., de 0.30 a 0.25) para aumentar la sensibilidad del modelo de manera inmediata. | 
| **Cambio en la distribución de variables** | Índice de Estabilidad de Población (PSI) > 0.25 o prueba Kolmogorov-Smirnov p-valor < 0.01 en variables clave | **Reentrenar** | MLOps Engineer / Data Engineer | Actualizar los parámetros del pipeline (`StandardScaler`, imputadores) y reentrenar el modelo con el nuevo perfil de clientes. | 
| **Aparición de nuevas categorías** | Nuevos valores no registrados en el pipeline para métodos de pago, planes o tipos de servicio | **Revisar** | Data Engineer / MLOps Engineer | Auditar el origen de datos, actualizar la codificación del `ColumnTransformer` (`OneHotEncoder`/`OrdinalEncoder`) y validar la ingesta. | 
| **Pérdida de disponibilidad** | Falla masiva en la API de inferencia o tasa de datos nulos/faltantes > 20% en la entrada | **Suspender** | MLOps Engineer / Equipo de Infraestructura | Activar el *Kill Switch* para detener la inferencia automática y conmutar a una regla de reserva (*fallback*) basada en heurísticas de negocio. | 
| **Diferencias excesivas entre grupos** | Coeficiente de Impacto Dispar (*Disparate Impact Ratio*) fuera del rango [0.80, 1.25] entre segmentos | **Recalibrar** | Responsable de Gobierno de Datos / Data Scientist | Auditar la equidad del modelo por segmentos de clientes y aplicar algoritmos de recalibración o mitigación de sesgos. | 
| **Cambio en el proceso comercial** | Ajustes significativos en precios, campañas masivas de retención o nuevas políticas de permanencia | **Recalibrar / Reentrenar** | Product Owner / Data Scientist | Recalibrar las probabilidades predichas para reflejar el nuevo entorno comercial o reentrenar incorporando las nuevas ofertas. | 
| **Modificación de la definición de *Churn*** | Redefinición del concepto de abandono (p. ej., pasar de 30 días sin pago a cancelación explícita de contrato) | **Reentrenar desde cero** | Lead Data Scientist / Product Owner | Redefinir la variable objetivo \(y\), volver a etiquetar el histórico de datos y reejecutar todo el proceso de modelado y validación. |

# Paso 30: Documentar la trazabilidad final

| Elemento | Detalle |
| :--- | :--- |
| Versión del dataset | Telco Customer Churn (7,043 registros, desbalance 73.46% / 26.54%) |
| Fuente | Repositorio Kaggle (`blastchar/telco-customer-churn`) cargado vía `kagglehub`|
| Fecha de consulta | 25–31 de agosto de 2026 |
| Variables utilizadas | 20 variables originales más 2 generadas por ingeniería (`Total_Servicios`, `Cliente_Nuevo`) |
| Variables excluidas | `customerID` (identificador sin valor predictivo) |
| Pipeline | `ColumnTransformer` con imputación (mediana/moda), `StandardScaler` (numéricas), `OneHotEncoder` (nominales), `OrdinalEncoder` (binarias y ordinales), y `remainder='drop'` |
| Modelos evaluados | DummyClassifier (Baseline), Regresión Logística, Árbol de Decisión, RandomForestClassifier, SVC (Lineal, RBF, RBF Complejo), MLPClassifier |
| Hiperparámetros | Regresión Logística con `max_iter=1000`, `solver='lbfgs'`, `random_state=42` |
| Métricas de validación | Validación cruzada 5x3 (Regresión Logística): ROC-AUC 0.8489, Recall 0.5382, Precision 0.6687, F1-Score 0.5956, Accuracy 0.8065 |
| Métricas de prueba | Conjunto Test: ROC-AUC 0.8457[cite: 12]. Con umbral ajustado a 0.30: Recall 0.7620, Precision 0.5337, F1-score 0.6278 |
| Umbral | 0.30 (Ajustado desde 0.50 para priorizar Recall sin saturar campañas) |
| Modelo final | Regresión Logística[cite: 12] |
| Archivo serializado | `pipeline_final_churn.joblib`, `modelo_logreg_churn.joblib`, `umbral_churn.json` |
| Bibliotecas | `scikit-learn`, `pandas`, `numpy`, `joblib`, `shap`, `matplotlib`, `json` |
| Fecha de entrenamiento | 27 de septiembre de 2026 |
| Archivo principal | `predictor.py` para mecanismo de inferencia |
| Responsable | Equipo de Panditas Panaderos |

# Paso 31: Elaborar la ficha del modelo

**Nombre y versión**  
Predictor de Abandono de Clientes (Churn) v1.0

**Propósito**  
Predecir la probabilidad de fuga de clientes basada en su historial demográfico y de facturación, permitiendo priorizar recursos y ejecutar campañas de retención proactivas.

**Usuarios previstos**  
Especialistas en retención (CX), ejecutivos de cuenta, departamentos de marketing y áreas financieras.

**Usos permitidos**  
*   Identificar clientes de alto riesgo para canalizarlos a atención personalizada o campañas preventivas.
*   Analizar coeficientes de Regresión Logística para entender factores de riesgo estructurales del negocio.
*   Simular escenarios de costo-beneficio alternando el umbral de probabilidad operativa.

**Usos no recomendados**  
*   Automatizar restricciones de servicio, cancelaciones punitivas o alteraciones tarifarias sin intervención humana.
*   Atribuir causalidad estricta a factores aislados (por ejemplo, asumir irremediablemente que la fibra óptica causa cancelaciones sin contexto comercial).

**Dataset**  
Telco Customer Churn original de Kaggle (7,043 instancias). Se añadieron las variables temporales y de densidad `Cliente_Nuevo` y `Total_Servicios` para fortalecer la predicción.

**Variable objetivo**  
`Churn` (Clasificación binaria: 1 = Cancelación, 0 = Retención).

**Métricas**  
*   **ROC-AUC (Prueba):** 0.8457
*   **Recall Operativo (Umbral 0.30):** 0.7620
*   **Precision Operativa (Umbral 0.30):** 0.5337

**Umbral**  
**0.30**. Desciende del valor nominal para rescatar a la mayor cantidad posible de usuarios en riesgo, asumiendo una porción controlada de falsas alarmas (249 Falsos Positivos).

**Limitaciones**  
Falta de visibilidad sobre métricas transaccionales dinámicas (quejas recientes o fallas de servicio)[cite: 13]. Esto ocasiona que perfiles con alta antigüedad sean catalogados indebidamente como inquebrantables, generando Falsos Negativos si su razón de abandono es repentina.

**Riesgos**  
*   **Sobrecarga operativa (Falsos Positivos):** Enviar campañas a usuarios leales por sobreestimación de riesgo (17.7% de incidencias en pruebas), lo que puede saturar al equipo si no se segmenta.
*   **Degradación del modelo (Data Drift):** Evoluciones no rastreadas en precios de mercado, catálogos de planes o irrupciones de competidores que alteren silenciosamente el comportamiento general de la base.

**Análisis de sesgos**  
El ciclo de vida del cliente sesga la confianza estadística. En usuarios recientes (≤ 6 meses), el motor predice fugas masivamente debido a altas tasas de abandono histórico (55.63%), logrando un Recall de 0.8728 pero una tasa de Falsos Positivos de 0.5942. Por el contrario, en clientes maduros (> 6 meses), omite 1 de cada 3 fugas reales (FNR 0.3333) al apoyarse desproporcionadamente en su lealtad pasada.

**Mecanismo de inferencia**  
Aplicación terminal `predictor.py` que ingiere archivos JSON o diccionarios, aplicando de forma estricta los `ColumnTransformer` serializados. Incluye bloqueos programáticos (controles de calidad) si detecta variables ausentes, datos no numéricos mal formados o categorías inéditas antes de generar la alerta de riesgo.

**Plan de monitoreo**  
*   **Inspección Continua (Diaria):** Tiempos de latencia por debajo de los 2 segundos, umbrales de nulidad en variables raíz (`tenure`, `TotalCharges` > 5%) y categorías emergentes.
*   **Seguimiento de Distribuciones (Semanal):** Variaciones de mercado (Data Drift) > 10% en indicadores críticos monetarios y contractuales.
*   **Auditoría de Resultados (Mensual):** Descenso comprobado del Recall general por debajo del umbral base (0.70).

**Criterios de actualización**  
*   **Reentrenamiento:** Ante caída relativa > 10% en el Recall en tres evaluaciones consecutivas, o si ocurren cambios en la distribución poblacional (Índice de Estabilidad PSI > 0.25 o p-valor KS < 0.01).
*   **Ajuste de Umbral:** Descenso manual del punto de corte operativo ante incrementos no planificados de Falsos Negativos.
*   **Recalibración / Revisión:** Alteración en el esquema de precios, campañas masivas, políticas de permanencia, aparición de categorías nuevas en los datos, o disparidades de desempeño entre segmentos (Impacto Dispar fuera de 0.80 - 1.25).
*   **Reentrenamiento Total:** Si ocurre una modificación en la regla de negocio que define el estado de *Churn*.
*   **Suspensión (*Kill Switch*):** Fallas masivas en la API de inferencia o tasas de campos nulos superiores al 20% en entrada.

**Responsable de revisión**  
Equipo de Panditas Panaderos